# Host-association analyses

## 1. Configuration used by the shared sensitivity-analysis workflow

In [ ]:
#config.yaml
batch_dir: E:/Data/Ralstonia/Pycharm/graduation2/new_analysis/SHAP/teacher_rerun_batch2_n10

strain_id: Accession
tree_file: E:/Data/Ralstonia/Pycharm/graduation2/new_analysis/SHAP/teacher_rerun_batch2_n10/data/RSSC.nwk
metadata_file: E:/Data/Ralstonia/Pycharm/graduation2/new_analysis/SHAP/teacher_rerun_batch2_n10/data/Ralstonia.biovar.data.20260813.tsv

trait_columns:
  disaccharide: disaccharide_use
  hexose_alcohol: hexose_alcohol_use

trait_source_columns:
  disaccharide_source: disaccharide_source
  hexose_alcohol_source: hexose_alcohol_source

habitat_columns:
  isolation_source: host
  host_species: host_species_clean
  host_family: host_family_clean
  host_major_clade: host_major_clade

metadata_columns:
  phylotype: phylotype
  country: Country
  continent: Continent
  bioproject: GenBank_BioProject

settings:
  zero_branch_length_replacement: 1e-10
  top_host_species_n: 10
  min_host_species_n: 10
  min_host_family_n: 10
  min_host_family_n_sensitivity: [5, 10, 20]
  top_host_species_sensitivity: [5, 10, 15]
  phylo_pc_axes_primary: 5
  phylo_pc_axes_sensitivity: [3, 5, 10]
  permutations: 999
  stochastic_maps: 500
  bootstrap_replicates: 500
  seed: 20260815
  n_cores: 1
  mcmc:
    nitt: 300000
    burnin: 100000
    thin: 200
  co_transition_near_distance: 2x_median_branch_length


## 2. General helper functions

In [ ]:
suppressPackageStartupMessages({
  library(yaml)
  library(readr)
  library(dplyr)
  library(tidyr)
  library(tibble)
  library(purrr)
  library(ape)
  library(phylolm)
})

read_config <- function(path = NULL) {
  if (is.null(path)) {
    path <- Sys.getenv("CODEX_CFG", unset = "config.yaml")
  }
  cfg <- yaml::read_yaml(path)
  bd <- normalizePath(cfg$batch_dir, winslash = "/", mustWork = FALSE)
  cfg$batch_dir <- bd
  cfg$paths <- list(
    root = bd,
    results = file.path(bd, "results"),
    tables = file.path(bd, "results", "tables"),
    models = file.path(bd, "results", "models"),
    pcs = file.path(bd, "results", "phylo_correction_sensitivity"),
    evolutionary = file.path(bd, "results", "evolutionary"),
    sensitivity = file.path(bd, "results", "sensitivity"),
    figures = file.path(bd, "figures"),
    report = file.path(bd, "report")
  )
  for (p in unlist(cfg$paths, use.names = FALSE)) {
    dir.create(p, recursive = TRUE, showWarnings = FALSE)
  }
  cfg
}

load_curated_metadata <- function(cfg) {
  read_tsv(file.path(cfg$paths$tables, "curated_metadata.tsv"),
           col_types = cols(.default = col_character()))
}

load_analysis_tree <- function(cfg) {
  ape::read.tree(file.path(cfg$paths$models, "analysis_tree.nwk"))
}

prune_to_strains <- function(tree, data) {
  dd <- data %>%
    mutate(strain_id = stringr::str_trim(as.character(strain_id))) %>%
    filter(!is.na(strain_id), strain_id %in% tree$tip.label) %>%
    distinct(strain_id, .keep_all = TRUE)
  tr <- ape::keep.tip(tree, dd$strain_id)
  dd <- as.data.frame(dd[match(tr$tip.label, dd$strain_id), , drop = FALSE])
  rownames(dd) <- dd$strain_id
  list(data = dd, tree = tr)
}

fit_phyloglm <- function(formula, data, tree) {
  warn <- character()
  fit <- tryCatch(
    withCallingHandlers(
      phylolm::phyloglm(formula, data = data, phy = tree,
                        method = "logistic_MPLE", boot = 0),
      warning = function(w) {
        warn <<- c(warn, conditionMessage(w))
        invokeRestart("muffleWarning")
      }
    ),
    error = function(e) e
  )
  if (inherits(fit, "error")) {
    warn_msg <- paste(unique(warn), collapse = " | ")
    warn_msg <- gsub("[\\r\\n\\t]+", " ", warn_msg, perl = TRUE)
    if (!nzchar(trimws(warn_msg))) warn_msg <- NA_character_
    return(list(n = NA_integer_, ok = FALSE, logLik = NA_real_, coefs = NULL,
                alpha = NA_real_, converged = NA,
                error = conditionMessage(fit),
                warnings = warn_msg))
  }
  co <- tryCatch(summary(fit)$coefficients, error = function(e) NULL)
  warn_msg <- paste(unique(warn), collapse = " | ")
  warn_msg <- gsub("[\\r\\n\\t]+", " ", warn_msg, perl = TRUE)
  if (!nzchar(trimws(warn_msg))) warn_msg <- NA_character_
  list(n = fit$n, ok = TRUE, logLik = fit$logLik, coefs = co,
       alpha = if (!is.null(fit$alpha)) fit$alpha else NA_real_,
       converged = if (!is.null(fit$converged)) fit$converged else NA,
       error = NULL,
       warnings = warn_msg)
}

extract_phyloglm_row <- function(fit, predictor, trait, level) {
  row <- tibble(
    metabolic_trait = trait,
    ecological_level = level,
    predictor = predictor,
    n = NA_integer_,
    phyloglm_beta = NA_real_, phyloglm_se = NA_real_,
    phyloglm_OR = NA_real_, phyloglm_CI_low = NA_real_,
    phyloglm_CI_high = NA_real_, phyloglm_P = NA_real_,
    phyloglm_alpha = NA_real_, phyloglm_converged = NA,
    phyloglm_error = NA_character_, phyloglm_warnings = NA_character_
  )
  if (is.null(fit) || !isTRUE(fit$ok) || is.null(fit$coefs)) {
    row$phyloglm_error <- if (is.null(fit)) "fit missing" else fit$error
    if (is.null(row$phyloglm_error) || is.na(row$phyloglm_error)) {
      row$phyloglm_error <- "fit failed"
    }
    row$phyloglm_warnings <- if (!is.null(fit)) fit$warnings else NA_character_
    return(row)
  }
  row$n <- fit$n
  row$phyloglm_alpha <- fit$alpha
  row$phyloglm_converged <- fit$converged
  row$phyloglm_warnings <- if (isTRUE(nzchar(fit$warnings))) fit$warnings else NA_character_
  co <- fit$coefs
  if (!(predictor %in% rownames(co))) {
    row$phyloglm_error <- "coefficient_missing"
    return(row)
  }
  pcol <- intersect(c("p.value", "Pr(>|z|)", "Pr(>z)"), colnames(co))[1]
  b <- unname(co[predictor, "Estimate"])
  se <- unname(co[predictor, "StdErr"])
  row$phyloglm_beta <- b
  row$phyloglm_se <- se
  row$phyloglm_OR <- exp(b)
  row$phyloglm_CI_low <- exp(b - qnorm(0.975) * se)
  row$phyloglm_CI_high <- exp(b + qnorm(0.975) * se)
  row$phyloglm_P <- if (!is.na(pcol)) unname(co[predictor, pcol]) else NA_real_
  row
}

add_fdr <- function(df, pcol, groupcol) {
  df %>%
    group_by(.data[[groupcol]]) %>%
    mutate(fdr = p.adjust(.data[[pcol]], method = "BH")) %>%
    ungroup()
}

binomial_ci <- function(x, n) {
  if (is.na(n) || n == 0) return(c(NA_real_, NA_real_))
  p <- x / n
  se <- sqrt(p * (1 - p) / n)
  lower <- max(0, p - 1.96 * se)
  upper <- min(1, p + 1.96 * se)
  if (p == 0) upper <- 1 - 0.05^(1 / n)
  if (p == 1) lower <- 0.05^(1 / n)
  c(lower, upper)
}

detect_separation <- function(fit) {
  ds <- tryCatch(logistf::detect_separation(fit), error = function(e) NULL)
  if (!is.null(ds)) {
    return(isTRUE(ds$out))
  }
  co <- tryCatch(coef(fit), error = function(e) NULL)
  !is.null(co) && (any(!is.finite(co)) || any(abs(co) > 12))
}

`%||%` <- function(x, y) {
  if (is.null(x)) y else x
}


## 3. GLMM helper functions

In [ ]:
suppressPackageStartupMessages({
  library(brms)
  library(rstan)
  library(dplyr)
  library(tibble)
})

save_pars <- function(all = FALSE) {
  list(all = all)
}

fit_brms_glmm <- function(dd, K, trait, predictor, seed,
                          iter = 2000, warmup = 1000,
                          chains = 4, cores = 4) {
  dd$strain <- rownames(dd)
  dd[[trait]] <- as.integer(dd[[trait]])
  fml <- as.formula(paste(trait, "~", predictor, "+ (1 | gr(strain, cov = K))"))
  fit <- tryCatch(
    brms::brm(
      fml, data = dd, data2 = list(K = K),
      family = brms::bernoulli(link = "logit"),
      prior = brms::set_prior("normal(0, 5)", class = "b"),
      iter = iter, warmup = warmup, chains = chains, cores = cores,
      seed = seed, refresh = 0,
      control = list(adapt_delta = 0.95, max_treedepth = 12)
    ),
    error = function(e) NULL
  )
  if (is.null(fit)) return(list(ok = FALSE, error = "brms fit failed"))
  list(ok = TRUE, fit = fit,
       summary = tryCatch(summary(fit), error = function(e) NULL))
}

fit_one_glmm <- function(dd, K, trait, seed,
                         iter = 2000, warmup = 1000,
                         chains = 4, cores = 4) {
  dd$strain <- rownames(dd)
  dd[[trait]] <- as.integer(dd[[trait]])
  fml <- as.formula(paste(trait, "~ x + (1 | gr(strain, cov = K))"))
  fit <- tryCatch(
    brms::brm(
      fml, data = dd, data2 = list(K = K),
      family = brms::bernoulli(link = "logit"),
      prior = brms::set_prior("normal(0, 5)", class = "b"),
      iter = iter, warmup = warmup, chains = chains, cores = cores,
      seed = seed, refresh = 0,
      control = list(adapt_delta = 0.95, max_treedepth = 12)
    ),
    error = function(e) NULL
  )
  if (is.null(fit)) return(list(ok = FALSE, error = "brms fit failed"))
  list(ok = TRUE, fit = fit,
       summary = tryCatch(summary(fit), error = function(e) NULL))
}

extract_glmm_row <- function(res, trait, contrast, level, test_id) {
  row <- tibble(
    metabolic_trait = trait, ecological_level = level, contrast = contrast,
    test_id = test_id,
    glmm_beta = NA_real_, glmm_OR = NA_real_, glmm_CI_low = NA_real_,
    glmm_CI_high = NA_real_, glmm_posterior_prob_beta_gt0 = NA_real_,
    phylogenetic_variance = NA_real_, glmm_Rhat = NA_real_,
    glmm_Bulk_ESS = NA_real_, glmm_Tail_ESS = NA_real_,
    glmm_divergences = NA_integer_, glmm_ok = FALSE,
    glmm_error = if (is.null(res) || !isTRUE(res$ok)) "fit failed" else NA_character_
  )
  if (is.null(res) || !isTRUE(res$ok) || is.null(res$summary)) return(row)
  fx <- res$summary$fixed
  if (!"x" %in% rownames(fx)) {
    row$glmm_error <- "coef not found"
    return(row)
  }
  r <- fx["x", ]
  post <- posterior::as_draws_df(res$fit)
  ppos <- if ("b_x" %in% colnames(post)) mean(post$b_x > 0) else NA_real_
  ranefs <- res$summary$random
  phy_var <- if (length(ranefs) > 0 && nrow(ranefs[[1]]) > 0) {
    ranefs[[1]][1, "Estimate"]^2
  } else NA_real_
  sp <- tryCatch(rstan::get_sampler_params(res$fit$fit, inc_warmup = FALSE),
                 error = function(e) NULL)
  div <- if (is.null(sp)) NA_integer_ else sum(unlist(lapply(sp, function(x) x[, "divergent__"])))
  tibble(
    metabolic_trait = trait, ecological_level = level, contrast = contrast,
    test_id = test_id,
    glmm_beta = r[["Estimate"]], glmm_OR = exp(r[["Estimate"]]),
    glmm_CI_low = exp(r[["l-95% CI"]]), glmm_CI_high = exp(r[["u-95% CI"]]),
    glmm_posterior_prob_beta_gt0 = ppos,
    phylogenetic_variance = phy_var,
    glmm_Rhat = r[["Rhat"]], glmm_Bulk_ESS = r[["Bulk_ESS"]],
    glmm_Tail_ESS = r[["Tail_ESS"]], glmm_divergences = div,
    glmm_ok = TRUE, glmm_error = NA_character_
  )
}

extract_glmm_row2 <- function(res, trait, contrast, level, predictor, test_id) {
  row <- tibble(
    metabolic_trait = trait, ecological_level = level, contrast = contrast,
    test_id = test_id,
    glmm_beta = NA_real_, glmm_OR = NA_real_, glmm_CI_low = NA_real_,
    glmm_CI_high = NA_real_, glmm_posterior_prob_beta_gt0 = NA_real_,
    phylogenetic_variance = NA_real_, glmm_Rhat = NA_real_,
    glmm_Bulk_ESS = NA_real_, glmm_Tail_ESS = NA_real_,
    glmm_divergences = NA_integer_, glmm_ok = FALSE,
    glmm_error = if (is.null(res) || !isTRUE(res$ok)) "fit failed" else NA_character_
  )
  if (is.null(res) || !isTRUE(res$ok) || is.null(res$summary)) return(row)
  fx <- res$summary$fixed
  if (!predictor %in% rownames(fx)) {
    row$glmm_error <- "coef not found"
    return(row)
  }
  r <- fx[predictor, ]
  post <- posterior::as_draws_df(res$fit)
  bname <- paste0("b_", predictor)
  ppos <- if (bname %in% colnames(post)) mean(post[[bname]] > 0) else NA_real_
  ranefs <- res$summary$random
  phy_var <- if (length(ranefs) > 0 && nrow(ranefs[[1]]) > 0) {
    ranefs[[1]][1, "Estimate"]^2
  } else NA_real_
  sp <- tryCatch(rstan::get_sampler_params(res$fit$fit, inc_warmup = FALSE),
                 error = function(e) NULL)
  div <- if (is.null(sp)) NA_integer_ else sum(unlist(lapply(sp, function(x) x[, "divergent__"])))
  tibble(
    metabolic_trait = trait, ecological_level = level, contrast = contrast,
    test_id = test_id,
    glmm_beta = r[["Estimate"]], glmm_OR = exp(r[["Estimate"]]),
    glmm_CI_low = exp(r[["l-95% CI"]]), glmm_CI_high = exp(r[["u-95% CI"]]),
    glmm_posterior_prob_beta_gt0 = ppos,
    phylogenetic_variance = phy_var,
    glmm_Rhat = r[["Rhat"]], glmm_Bulk_ESS = r[["Bulk_ESS"]],
    glmm_Tail_ESS = r[["Tail_ESS"]], glmm_divergences = div,
    glmm_ok = TRUE, glmm_error = NA_character_
  )
}


## 4.Prevalence estimates

In [ ]:
import numpy as np
import pandas as pd

file_path = r"E:\Data\Ralstonia\Pycharm\graduation2\new_analysis\association\mapping_with_metabolism_species_reclassified_broad_source.csv"
raw = pd.read_csv(file_path, encoding="gb18030")

# broad_source 中的 dicotyledon / monocotyledon 用于植物分层；
# Soil/rhizosphere 和 Water 单独作为 isolation source 面板。
plant_data = raw.loc[
    raw["broad_source"].isin(["monocotyledon", "dicotyledon"])
    & raw["host_norm"].notna()
    & raw["family"].notna()
].copy()

trait_cols = {
    "Disaccharide utilization": "disaccharide utilization",
    "Hexitol utilization": "hexitol utilization",
}

source_data = raw.loc[
    raw["broad_source"].isin(["soil/rhizosphere", "water"])
].copy()

level_cols = {
    "broad_source": "broad_source",
    "species": "host_norm",
    "family": "family",
    "cotyledon": "broad_source",
}

data_by_level = {
    "broad_source": source_data,
    "species": plant_data,
    "family": plant_data,
    "cotyledon": plant_data,
}

min_n_by_level = {
    "broad_source": 1,
    "species": 10,
    "family": 10,
    "cotyledon": 1,
}


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan

    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half_width = z * np.sqrt(
        p * (1 - p) / n + z**2 / (4 * n**2)
    ) / denom

    return max(0, center - half_width), min(1, center + half_width)


rows = []

for level, group_col in level_cols.items():
    level_data = data_by_level[level]
    min_n = min_n_by_level[level]

    for label, g in level_data.groupby(group_col, dropna=True):
        n = len(g)

        if n < min_n:
            continue

        for trait_name, trait_col in trait_cols.items():
            positive = int(g[trait_col].sum())
            negative = n - positive
            prevalence = positive / n
            ci_low, ci_high = wilson_ci(positive, n)

            rows.append({
                "level": level,
                "label": label,
                "trait": trait_name,
                "n": n,
                "positive": positive,
                "negative": negative,
                "prevalence": prevalence,
                "ci_low": ci_low,
                "ci_high": ci_high,
            })

prevalence_ci_results = pd.DataFrame(rows)

prevalence_ci_results = prevalence_ci_results.sort_values(
    ["level", "n", "label", "trait"],
    ascending=[True, False, True, True],
).reset_index(drop=True)

print("plant samples:", len(plant_data))
print("environmental samples (soil + water):", len(source_data))
print("result rows:", len(prevalence_ci_results))
print(prevalence_ci_results)

## 5.host-family and host-species lists (n >= 10)

In [ ]:
# Prepare the n >= 10 family and species lists used by the phyloglm subsets
cfg <- read_config()
dat <- load_curated_metadata(cfg)

eligible_families <- dat %>%
  filter(source_broad == "plant", !is.na(host_family_clean)) %>%
  count(host_family_clean, name = "n") %>%
  filter(n >= cfg$settings$min_host_family_n)

eligible_species <- dat %>%
  filter(source_broad == "plant", !is.na(host_species_clean),
         host_species_clean != "unknown") %>%
  count(host_species_clean, name = "n") %>%
  filter(n >= cfg$settings$min_host_species_n)

write_tsv(eligible_families,
          file.path(cfg$paths$tables, "eligible_host_families.tsv"))
write_tsv(eligible_species,
          file.path(cfg$paths$tables, "top_host_species.tsv"))

## 6. Primary Ives-Garland phylogenetic logistic regression

In [ ]:
.libPaths(c("D:/Rlib", .libPaths()))

suppressPackageStartupMessages({
  library(ape)
  library(phylolm)
  library(dplyr)
  library(readr)
  library(stringr)
  library(tibble)
  library(purrr)
})

base_dir <- "E:/Data/Ralstonia/Pycharm/graduation2/new_analysis/association"
input_csv <- file.path(base_dir, "mapping_with_metabolism_species_reclassified.csv")
tree_file <- file.path(base_dir, "RSSC_biovar_分享包/05_模型与数据_Models/analysis_tree.nwk")
out_dir <- file.path(base_dir, "final_phyloglm_primary_newdata")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

set.seed(20260815)

read_tree_clean <- function(path, epsilon = 1e-10) {
  txt <- paste(readLines(path, warn = FALSE, encoding = "UTF-8"), collapse = "")
  txt <- gsub("\\s+", "", txt)
  tr <- ape::read.tree(text = txt)
  tr$tip.label <- stringr::str_trim(tr$tip.label)
  if (is.null(tr$edge.length)) {
    tr$edge.length <- rep(epsilon, nrow(tr$edge))
  }
  tr$edge.length[is.na(tr$edge.length) | tr$edge.length <= 0] <- epsilon
  tr
}

read_metadata <- function(path) {
  raw <- readr::read_csv(
    path,
    locale = readr::locale(encoding = "GB18030"),
    show_col_types = FALSE,
    na = c("", "NA", "NaN")
  )

  txt <- stringr::str_to_lower(paste(raw$Host %||% "", raw$Isolation_source %||% ""))
  plant <- !is.na(raw$cotyledon) & raw$cotyledon %in% c("monocotyledon", "dicotyledon")
  water <- stringr::str_detect(txt, "water|river|stream|lake|pond|canal|irrigation|wastewater")
  soil <- stringr::str_detect(txt, "soil|rhizosphere|substrate|mud|sediment")

  raw %>%
    transmute(
      strain_id = stringr::str_trim(as.character(Accession)),
      GenBank_BioProject = GenBank_BioProject,
      Country = Parsed_Country,
      Continent = Parsed_Continent,
      phylotype = phylotype,
      host_species_clean = na_if(stringr::str_trim(as.character(host_norm)), ""),
      host_family_clean = na_if(stringr::str_trim(as.character(family)), ""),
      host_major_clade = dplyr::case_when(
        cotyledon == "monocotyledon" ~ "monocot",
        cotyledon == "dicotyledon" ~ "eudicot",
        TRUE ~ NA_character_
      ),
      source_broad = dplyr::case_when(
        plant ~ "plant",
        water & !plant ~ "water",
        soil & !plant & !water ~ "soil/rhizosphere",
        TRUE ~ NA_character_
      ),
      disaccharide_use = as.integer(`disaccharide utilization`),
      hexose_alcohol_use = as.integer(`hexitol utilization`),
      utilization_source = `utilization source`
    ) %>%
    mutate(
      host_species_clean = if_else(host_species_clean %in% c("unknown", "Unknown", "missing"), NA_character_, host_species_clean),
      host_family_clean = if_else(host_family_clean %in% c("unknown", "Unknown", "missing"), NA_character_, host_family_clean),
      host_monocot = case_when(
        host_major_clade == "monocot" ~ 1L,
        host_major_clade == "eudicot" ~ 0L,
        TRUE ~ NA_integer_
      )
    )
}

`%||%` <- function(x, y) {
  if (is.null(x)) y else x
}

prune_to_strains <- function(tree, data) {
  dd <- data %>%
    mutate(strain_id = stringr::str_trim(as.character(strain_id))) %>%
    filter(!is.na(strain_id), strain_id %in% tree$tip.label) %>%
    distinct(strain_id, .keep_all = TRUE)
  tr <- ape::keep.tip(tree, dd$strain_id)
  dd <- as.data.frame(dd[match(tr$tip.label, dd$strain_id), ])
  rownames(dd) <- dd$strain_id
  list(data = dd, tree = tr)
}

fit_phyloglm_safe <- function(formula, data, tree) {
  warnings <- character()
  fit <- tryCatch(
    withCallingHandlers(
      phylolm::phyloglm(
        formula,
        data = data,
        phy = tree,
        method = "logistic_MPLE",
        boot = 0
      ),
      warning = function(w) {
        warnings <<- c(warnings, conditionMessage(w))
        invokeRestart("muffleWarning")
      }
    ),
    error = function(e) e
  )
  attr(fit, "captured_warnings") <- paste(unique(warnings), collapse = " | ")
  fit
}

extract_row <- function(fit, predictor, trait, level) {
  row <- tibble(
    metabolic_trait = trait,
    ecological_level = level,
    predictor = predictor,
    n = NA_integer_,
    phyloglm_beta = NA_real_,
    phyloglm_se = NA_real_,
    phyloglm_OR = NA_real_,
    phyloglm_CI_low = NA_real_,
    phyloglm_CI_high = NA_real_,
    phyloglm_P = NA_real_,
    phyloglm_alpha = NA_real_,
    phyloglm_converged = NA,
    phyloglm_error = NA_character_,
    phyloglm_warnings = attr(fit, "captured_warnings")
  )

  if (inherits(fit, "error")) {
    row$phyloglm_error <- conditionMessage(fit)
    return(row)
  }

  row$n <- fit$n
  row$phyloglm_alpha <- if (!is.null(fit$alpha)) fit$alpha else NA_real_
  row$phyloglm_converged <- if (!is.null(fit$converged)) fit$converged else NA

  co <- tryCatch(summary(fit)$coefficients, error = function(e) NULL)
  if (is.null(co) || !(predictor %in% rownames(co))) {
    row$phyloglm_error <- "coefficient_missing"
    return(row)
  }

  pcol <- intersect(c("p.value", "Pr(>|z|)", "Pr(>z)"), colnames(co))[1]
  beta <- unname(co[predictor, "Estimate"])
  se <- unname(co[predictor, "StdErr"])
  row$phyloglm_beta <- beta
  row$phyloglm_se <- se
  row$phyloglm_OR <- exp(beta)
  row$phyloglm_CI_low <- exp(beta - qnorm(0.975) * se)
  row$phyloglm_CI_high <- exp(beta + qnorm(0.975) * se)
  row$phyloglm_P <- if (!is.na(pcol)) unname(co[predictor, pcol]) else NA_real_
  row
}

mark_separation_na <- function(row) {
  row$phyloglm_beta <- NA_real_
  row$phyloglm_se <- NA_real_
  row$phyloglm_OR <- NA_real_
  row$phyloglm_CI_low <- NA_real_
  row$phyloglm_CI_high <- NA_real_
  row$phyloglm_P <- NA_real_
  row$phyloglm_alpha <- NA_real_
  row$phyloglm_error <- "complete_separation_focal_trait_invariant"
  row
}

run_phyloglm <- function(trait, predictor, data, tree, level, fdr_group, test_id,
                         contrast_label = NULL, focal_col = NULL,
                         global_P = NA_real_) {
  pr <- prune_to_strains(tree, data)
  dd <- pr$data
  tr <- pr$tree

  focal_n <- NA_integer_
  focal_prevalence <- NA_real_
  separation <- FALSE
  sparse <- FALSE
  if (!is.null(focal_col)) {
    foc <- dd[[focal_col]]
    if (is.numeric(foc) && any(foc == 1, na.rm = TRUE)) {
      focal_n <- sum(foc == 1, na.rm = TRUE)
      focal_prevalence <- mean(dd[[trait]][foc == 1], na.rm = TRUE)
      separation <- isTRUE(focal_prevalence %in% c(0, 1))
      sparse <- focal_n < 5 || (nrow(dd) - focal_n) < 5
    }
  }

  fit <- fit_phyloglm_safe(as.formula(paste(trait, "~", predictor)), dd, tr)
  row <- extract_row(fit, predictor, trait, level)
  if (isTRUE(separation)) row <- mark_separation_na(row)

  row$focal_n <- focal_n
  row$focal_prevalence <- focal_prevalence
  row$sparse <- sparse
  row$contrast <- contrast_label
  row$fdr_group <- fdr_group
  row$test_id <- test_id
  row$global_P <- global_P
  row$separation <- separation
  if (!inherits(fit, "error") && !is.null(fit$n)) row$n <- fit$n
  row
}

add_fdr <- function(df) {
  df %>%
    group_by(fdr_group) %>%
    mutate(fdr = p.adjust(phyloglm_P, method = "BH")) %>%
    ungroup()
}

global_lrt_p <- function(trait, formula_full, formula_null, data, tree, df_lrt) {
  pr <- prune_to_strains(tree, data)
  full <- fit_phyloglm_safe(as.formula(formula_full), pr$data, pr$tree)
  null <- fit_phyloglm_safe(as.formula(formula_null), pr$data, pr$tree)
  if (inherits(full, "error") || inherits(null, "error")) return(NA_real_)
  if (is.null(full$logLik) || is.null(null$logLik)) return(NA_real_)
  lrt <- 2 * (full$logLik - null$logLik)
  pchisq(lrt, df = df_lrt, lower.tail = FALSE)
}

dat <- read_metadata(input_csv)
tree <- read_tree_clean(tree_file, epsilon = 1e-10)

traits <- c("disaccharide_use", "hexose_alcohol_use")
out <- list()

input_summary <- tibble(
  item = c(
    "n_total_rows",
    "n_tree_tips",
    "n_source_plant",
    "n_source_soil_rhizosphere",
    "n_source_water",
    "n_plant_with_cotyledon",
    "species_min_n",
    "family_min_n",
    "tree_file",
    "metadata_file"
  ),
  value = c(
    nrow(dat),
    length(tree$tip.label),
    sum(dat$source_broad == "plant", na.rm = TRUE),
    sum(dat$source_broad == "soil/rhizosphere", na.rm = TRUE),
    sum(dat$source_broad == "water", na.rm = TRUE),
    sum(!is.na(dat$host_monocot)),
    10,
    10,
    tree_file,
    input_csv
  )
)
write_tsv(input_summary, file.path(out_dir, "input_summary.tsv"))
write_tsv(dat, file.path(out_dir, "metadata_used_for_phyloglm.tsv"))

# A. Broad isolation source: plant as reference; soil/rhizosphere and water one-vs-rest.
src <- dat %>%
  filter(source_broad %in% c("plant", "soil/rhizosphere", "water")) %>%
  mutate(source_broad = factor(source_broad, levels = c("plant", "soil/rhizosphere", "water")))

for (trt in traits) {
  gp <- global_lrt_p(trt, paste(trt, "~ source_broad"), paste(trt, "~ 1"), src, tree, df_lrt = 2)
  for (src_level in c("soil/rhizosphere", "water")) {
    dd <- src %>% mutate(focal_source = as.integer(source_broad == src_level))
    test_id <- paste0("src_", gsub("[^A-Za-z0-9_]", "_", src_level), "_", trt)
    out[[length(out) + 1]] <- run_phyloglm(
      trt, "focal_source", dd, tree, "broad_source",
      paste0("broad_source_", trt), test_id,
      contrast_label = paste0("source_broad", src_level),
      focal_col = "focal_source",
      global_P = gp
    )
  }
}

# B. Monocot vs eudicot.
plant_cot <- dat %>% filter(!is.na(host_monocot))
for (trt in traits) {
  test_id <- paste0("me_", trt)
  out[[length(out) + 1]] <- run_phyloglm(
    trt, "host_monocot", plant_cot, tree, "monocot_eudicot",
    paste0("monocot_eudicot_", trt), test_id,
    contrast_label = "host_monocot",
    focal_col = "host_monocot"
  )
}

# C. Host family: n >= 10, one-vs-rest among all plant strains with known family.
plant_family <- dat %>%
  filter(source_broad == "plant", !is.na(host_family_clean))
eligible_families <- plant_family %>%
  count(host_family_clean, name = "n") %>%
  filter(n >= 10) %>%
  arrange(host_family_clean)
write_tsv(eligible_families, file.path(out_dir, "eligible_host_families_n10.tsv"))

for (trt in traits) {
  for (fm in eligible_families$host_family_clean) {
    dd <- plant_family %>% mutate(focal_family = as.integer(host_family_clean == fm))
    test_id <- paste0("fam_", gsub("[^A-Za-z0-9_]", "_", fm), "_", trt)
    out[[length(out) + 1]] <- run_phyloglm(
      trt, "focal_family", dd, tree, paste0("family_", fm),
      paste0("family_", trt), test_id,
      contrast_label = fm,
      focal_col = "focal_family"
    )
  }
}

# D. Host species: n >= 10, one-vs-rest among all plant strains with known species.
plant_species <- dat %>%
  filter(source_broad == "plant", !is.na(host_species_clean))
eligible_species <- plant_species %>%
  count(host_species_clean, name = "n") %>%
  filter(n >= 10) %>%
  arrange(desc(n), host_species_clean)
write_tsv(eligible_species, file.path(out_dir, "eligible_host_species_n10.tsv"))

for (trt in traits) {
  for (sp in eligible_species$host_species_clean) {
    dd <- plant_species %>% mutate(focal_host = as.integer(host_species_clean == sp))
    test_id <- paste0("sp_", gsub("[^A-Za-z0-9_]", "_", sp), "_", trt)
    out[[length(out) + 1]] <- run_phyloglm(
      trt, "focal_host", dd, tree, paste0("species_", sp),
      paste0("species_", trt), test_id,
      contrast_label = sp,
      focal_col = "focal_host"
    )
  }
}

phylo_all <- bind_rows(out) %>%
  mutate(
    across(
      c(phyloglm_error, phyloglm_warnings, contrast, fdr_group, test_id),
      ~ stringr::str_squish(stringr::str_replace_all(as.character(.x), "[\r\n\t]+", " "))
    ),
    across(
      c(phyloglm_error, phyloglm_warnings, contrast, fdr_group, test_id),
      ~ dplyr::na_if(.x, "NA")
    )
  ) %>%
  select(
    metabolic_trait, ecological_level, predictor, n,
    phyloglm_beta, phyloglm_se, phyloglm_OR, phyloglm_CI_low,
    phyloglm_CI_high, phyloglm_P, phyloglm_alpha,
    phyloglm_converged, phyloglm_error, phyloglm_warnings,
    separation, focal_n, focal_prevalence, sparse, contrast,
    fdr_group, test_id, global_P
  ) %>%
  add_fdr()

write_tsv(phylo_all, file.path(out_dir, "phyloglm_association_models.tsv"))
write_tsv(phylo_all %>% filter(!is.na(fdr), fdr <= 0.05),
          file.path(out_dir, "phyloglm_association_models_q005.tsv"))
write_tsv(phylo_all %>% filter(separation),
          file.path(out_dir, "phyloglm_complete_separation_tests.tsv"))

readable <- phylo_all %>%
  mutate(
    trait_label = recode(metabolic_trait,
                         disaccharide_use = "Disaccharide use",
                         hexose_alcohol_use = "Hexose-alcohol use"),
    OR_CI = if_else(
      is.na(phyloglm_OR),
      NA_character_,
      sprintf("%.3f (%.3f-%.3f)", phyloglm_OR, phyloglm_CI_low, phyloglm_CI_high)
    ),
    P = if_else(is.na(phyloglm_P), NA_character_, formatC(phyloglm_P, format = "e", digits = 3)),
    q = if_else(is.na(fdr), NA_character_, formatC(fdr, format = "e", digits = 3))
  ) %>%
  select(trait_label, ecological_level, contrast, n, focal_n, focal_prevalence,
         OR_CI, P, q, separation, phyloglm_error, global_P)
write_tsv(readable, file.path(out_dir, "phyloglm_association_models_readable.tsv"))

cat("Final primary phyloglm finished.\n")
cat("Output:", out_dir, "\n")
cat("Rows:", nrow(phylo_all), "\n")
cat("FDR <= 0.05 rows:", sum(!is.na(phylo_all$fdr) & phylo_all$fdr <= 0.05), "\n\n")

cat("Eligible species n>=10:\n")
print(eligible_species, n = Inf)
cat("\nEligible families n>=10:\n")
print(eligible_families, n = Inf)

cat("\nSignificant rows (FDR <= 0.05):\n")
print(
  phylo_all %>%
    filter(!is.na(fdr), fdr <= 0.05) %>%
    select(metabolic_trait, ecological_level, contrast, n, focal_n,
           focal_prevalence, phyloglm_OR, phyloglm_P, fdr, separation),
  n = Inf
)

cat("\nComplete separation rows:\n")
print(
  phylo_all %>%
    filter(separation) %>%
    select(metabolic_trait, ecological_level, contrast, n, focal_n,
           focal_prevalence, phyloglm_error),
  n = Inf
)

## 7. Preparation of shared model subsets used by PC/MDS and kinship GLMM

In [ ]:
# 07 - Primary phylogenetic correction: Ives-Garland phylogenetic logistic
# regression (phyloglm) for all primary ecological comparisons
suppressPackageStartupMessages({
  source(file.path("R", "utils.R"))
})

cfg <- read_config()
set.seed(cfg$settings$seed)
dat <- load_curated_metadata(cfg)
tree <- load_analysis_tree(cfg)

dat <- dat %>%
  mutate(
    disaccharide_use = as.integer(disaccharide_use),
    hexose_alcohol_use = as.integer(hexose_alcohol_use),
    host_monocot = case_when(
      host_major_clade == "monocot" ~ 1L,
      host_major_clade == "eudicot" ~ 0L,
      TRUE ~ NA_integer_
    )
  )

traits <- c("disaccharide_use", "hexose_alcohol_use")
subsets <- list()
fits <- list()

run_phyloglm <- function(trait, predictor, data, level, fdr_group, test_id,
                         contrast_label = NULL, focal_col = NULL) {
  pr <- prune_to_strains(tree, data)
  dd <- pr$data
  tr <- pr$tree
  subsets[[test_id]] <<- dd

  sep_info <- list(focal_n = NA_integer_, focal_prevalence = NA_real_,
                   separation = FALSE, sparse = FALSE)
  if (!is.null(focal_col)) {
    foc <- dd[[focal_col]]
    if (is.numeric(foc) && any(foc == 1)) {
      fp <- mean(dd[[trait]][foc == 1])
      sep_info$focal_n <- sum(foc == 1)
      sep_info$focal_prevalence <- fp
      sep_info$separation <- fp %in% c(0, 1)
      sep_info$sparse <- sep_info$focal_n < 5 || (nrow(dd) - sep_info$focal_n) < 5
    }
  }

  fit <- fit_phyloglm(as.formula(paste(trait, "~", predictor)), dd, tr)
  fits[[test_id]] <<- fit
  row <- extract_phyloglm_row(fit, predictor, trait, level)
  if (isTRUE(sep_info$separation)) {
    # MPLE is not identifiable under complete separation
    row <- row %>% mutate(
      phyloglm_beta = NA_real_, phyloglm_se = NA_real_,
      phyloglm_OR = NA_real_, phyloglm_CI_low = NA_real_,
      phyloglm_CI_high = NA_real_, phyloglm_P = NA_real_,
      phyloglm_alpha = NA_real_,
      phyloglm_error = "complete_separation_focal_trait_invariant"
      ,
      phyloglm_warnings = NA_character_
    )
  }
  row$focal_n <- sep_info$focal_n
  row$focal_prevalence <- sep_info$focal_prevalence
  row$separation <- sep_info$separation
  row$sparse <- sep_info$sparse
  row$contrast <- contrast_label
  row$fdr_group <- fdr_group
  row$test_id <- test_id
  row$n <- fit$n
  row
}

out <- list()

# --- A. Broad isolation source ---
src <- dat %>%
  filter(source_broad %in% c("plant", "soil/rhizosphere", "water")) %>%
  mutate(source_broad = factor(source_broad, levels = c("plant", "soil/rhizosphere", "water")))
for (tr in traits) {
  pr <- prune_to_strains(tree, src)
  fit_full <- fit_phyloglm(as.formula(paste(tr, "~ source_broad")), pr$data, pr$tree)
  fit_null <- fit_phyloglm(as.formula(paste(tr, "~ 1")), pr$data, pr$tree)
  global_P <- NA_real_
  if (fit_full$ok && fit_null$ok) {
    lrt <- 2 * (fit_full$logLik - fit_null$logLik)
    global_P <- pchisq(lrt, df = 2, lower.tail = FALSE)
  }
  src_bin <- list(
    list(level = "soil/rhizosphere", label = "source_broadsoil/rhizosphere",
         id = "src_soil_rhizosphere"),
    list(level = "water", label = "source_broadwater", id = "src_water"))
  rows <- map_dfr(src_bin, function(sb) {
    dd <- src %>% mutate(focal_source = as.integer(source_broad == sb$level))
    test_id <- paste0(sb$id, "_", tr)
    pr2 <- prune_to_strains(tree, dd)
    fit <- fit_phyloglm(as.formula(paste(tr, "~ focal_source")), pr2$data, pr2$tree)
    fits[[test_id]] <<- fit
    subsets[[test_id]] <<- pr2$data
    r <- extract_phyloglm_row(fit, "focal_source", tr, "broad_source")
    fp <- mean(pr2$data[[tr]][pr2$data$focal_source == 1])
    r$separation <- fp %in% c(0, 1)
    if (r$separation) {
      r <- r %>% mutate(
        phyloglm_beta = NA_real_, phyloglm_se = NA_real_,
        phyloglm_OR = NA_real_, phyloglm_CI_low = NA_real_,
        phyloglm_CI_high = NA_real_, phyloglm_P = NA_real_,
        phyloglm_alpha = NA_real_,
        phyloglm_error = "complete_separation_focal_trait_invariant"
        ,
        phyloglm_warnings = NA_character_
      )
    }
    r$focal_n <- sum(pr2$data$focal_source == 1)
    r$focal_prevalence <- fp
    r$sparse <- r$focal_n < 5 || (fit$n - r$focal_n) < 5
    r$contrast <- sb$label
    r$fdr_group <- paste0("broad_source_", tr)
    r$test_id <- test_id
    r$n <- fit$n
    r$global_P <- global_P
    r
  })
  if (fit_full$ok) {
    fits[[paste0("src_global_", tr)]] <- fit_full
  }
  out[[paste0("src_", tr)]] <- rows
}

# --- B. Monocot vs eudicot ---
plant <- dat %>% filter(!is.na(host_monocot))
for (tr in traits) {
  test_id <- paste0("me_", tr)
  out[[test_id]] <- run_phyloglm(tr, "host_monocot", plant, "monocot_eudicot",
                                  paste0("monocot_eudicot_", tr), test_id,
                                  contrast_label = "host_monocot",
                                  focal_col = "host_monocot")
}

# --- C. Host family (one-vs-rest vs all plant families) ---
fams <- read_tsv(file.path(cfg$paths$tables, "eligible_host_families.tsv"),
                 col_types = cols())
for (tr in traits) {
  rows <- map_dfr(fams$host_family_clean, function(fm) {
    dd <- dat %>% filter(source_broad == "plant", !is.na(host_family_clean)) %>%
      mutate(focal_family = as.integer(host_family_clean == fm))
    test_id <- paste0("fam_", gsub("[^A-Za-z0-9_]", "_", fm), "_", tr)
    run_phyloglm(tr, "focal_family", dd, paste0("family_", fm),
                 paste0("family_", tr), test_id, contrast_label = fm,
                 focal_col = "focal_family")
  })
  out[[paste0("fam_", tr)]] <- rows
}

# --- D. Top-10 host species (one-vs-rest vs all known plant hosts) ---
top_species <- read_tsv(file.path(cfg$paths$tables, "top_host_species.tsv"),
                        col_types = cols())
for (tr in traits) {
  rows <- map_dfr(top_species$host_species_clean, function(sp) {
    dd <- dat %>% filter(source_broad == "plant", !is.na(host_species_clean),
                         host_species_clean != "unknown") %>%
      mutate(focal_host = as.integer(host_species_clean == sp))
    test_id <- paste0("sp_", gsub("[^A-Za-z0-9_]", "_", sp), "_", tr)
    run_phyloglm(tr, "focal_host", dd, paste0("species_", sp),
                 paste0("species_", tr), test_id, contrast_label = sp,
                 focal_col = "focal_host")
  })
  out[[paste0("sp_", tr)]] <- rows
}

phylo_all <- bind_rows(out)
phylo_all <- add_fdr(phylo_all, "phyloglm_P", "fdr_group")
write_tsv(phylo_all, file.path(cfg$paths$tables, "phyloglm_association_models.tsv"))
saveRDS(subsets, file.path(cfg$paths$models, "phyloglm_subsets.rds"))
saveRDS(fits, file.path(cfg$paths$models, "phyloglm_fits.rds"))

cat("phyloglm models written; n =", nrow(phylo_all), "\n")
print(phylo_all %>%
        filter(fdr <= 0.05) %>%
        select(metabolic_trait, contrast, n, phyloglm_OR, phyloglm_CI_low,
               phyloglm_CI_high, phyloglm_P, phyloglm_alpha, fdr), n = 40)
cat("\nSeparated tests (phyloglm not interpreted):\n")
print(phylo_all %>% filter(separation) %>%
        select(metabolic_trait, contrast, focal_n, focal_prevalence))


## 8. Preparation of phylogenetic PC/MDS axes

In [ ]:
# 08 - Build phylogenetic PC/MDS ancestry axes for every phyloglm test subset
# Axes are chosen from phylogenetic structure only (patristic distances).
suppressPackageStartupMessages({
  source(file.path("R", "utils.R"))
})

cfg <- read_config()
set.seed(cfg$settings$seed)
tree <- load_analysis_tree(cfg)
subsets <- readRDS(file.path(cfg$paths$models, "phyloglm_subsets.rds"))

K_MAX <- 10
pco_list <- list()
sum_tbl <- list()

for (test_id in names(subsets)) {
  dd <- subsets[[test_id]]
  pr <- prune_to_strains(tree, dd)
  D <- ape::cophenetic.phylo(pr$tree)
  pco <- cmdscale(D, k = K_MAX, eig = TRUE)
  colnames(pco$points) <- paste0("PhyloPC", seq_len(ncol(pco$points)))
  ev <- pco$eig
  var_expl <- ev[seq_len(K_MAX)] / sum(ev[ev > 0])
  dat_pc <- cbind(pr$data, pco$points)
  pco_list[[test_id]] <- list(
    data = dat_pc,
    tree = pr$tree,
    eigenvalues = ev,
    variance_explained = var_expl,
    n_axes = K_MAX
  )
  sum_tbl[[test_id]] <- tibble(
    test_id = test_id,
    n = nrow(dat_pc),
    eig1 = ev[1], eig2 = ev[2], eig3 = ev[3],
    cumvar_3 = sum(var_expl[1:3]),
    cumvar_5 = sum(var_expl[1:5]),
    cumvar_10 = sum(var_expl[1:10])
  )
}

saveRDS(pco_list, file.path(cfg$paths$pcs, "phylogenetic_pcs.rds"))
write_tsv(bind_rows(sum_tbl), file.path(cfg$paths$pcs, "phylogenetic_pc_summary.tsv"))

cat("Phylogenetic PCs prepared for", length(pco_list), "subsets\n")
print(bind_rows(sum_tbl) %>% select(test_id, n, cumvar_3, cumvar_5, cumvar_10), n = 45)


## 9. PC/MDS-adjusted logistic regression

In [ ]:
# 09 - PC/MDS-adjusted logistic regression sensitivity analysis
# logit[P(trait=1)] = b0 + b1*ecological + g1*PhyloPC1 + ... + gK*PhyloPCK
suppressPackageStartupMessages({
  source(file.path("R", "utils.R"))
})

cfg <- read_config()
set.seed(cfg$settings$seed)
pco_list <- readRDS(file.path(cfg$paths$pcs, "phylogenetic_pcs.rds"))

K_primary <- cfg$settings$phylo_pc_axes_primary
K_sensitivity <- cfg$settings$phylo_pc_axes_sensitivity

logistf_control <- logistf::logistf.control(maxit = 1000, maxstep = 20)
pl_control <- logistf::logistpl.control(maxit = 1000, maxstep = 20)

fit_pc <- function(dd, trait, predictor, coef_name, K) {
  pc_cols <- paste0("PhyloPC", seq_len(K))
  fml <- as.formula(paste(trait, "~", predictor, "+",
                          paste(pc_cols, collapse = " + ")))
  warns <- character()
  glm_fit <- tryCatch(glm(fml, data = dd, family = binomial()),
                      error = function(e) NULL)
  use_logistf <- is.null(glm_fit) || detect_separation(glm_fit)
  if (!use_logistf) {
    co <- summary(glm_fit)$coefficients
    if (!coef_name %in% rownames(co)) return(NULL)
    b <- co[coef_name, "Estimate"]; se <- co[coef_name, "Std. Error"]
    p <- co[coef_name, "Pr(>|z|)"]
    ci <- c(b - qnorm(0.975) * se, b + qnorm(0.975) * se)
    method <- "glm"; sep_flag <- FALSE; ci_type <- "wald"
  } else {
    withCallingHandlers({
      lf <- tryCatch(
        logistf::logistf(fml, data = dd, pl = TRUE,
                         control = logistf_control,
                         plcontrol = pl_control),
        error = function(e) NULL)
    }, warning = function(w) {
      warns <<- c(warns, conditionMessage(w)); invokeRestart("muffleWarning")
    })
    if (is.null(lf)) return(NULL)
    co <- lf$coefficients
    if (coef_name %in% names(co)) {
      idx <- which(names(co) == coef_name)
    } else {
      idx <- ncol(model.matrix(fml, dd))
    }
    b <- co[idx]; se <- sqrt(diag(lf$var))[idx]; p <- lf$prob[idx]
    ci <- tryCatch({
      ci0 <- confint(lf, control = pl_control)
      if (coef_name %in% rownames(ci0)) ci0[coef_name, ] else ci0[nrow(ci0), ]
    }, error = function(e) rep(NA_real_, 2))
    if (any(!is.finite(ci))) {
      ci <- c(b - qnorm(0.975) * se, b + qnorm(0.975) * se)
      ci_type <- "wald_fallback"
    } else ci_type <- "profile"
    method <- "logistf"; sep_flag <- TRUE
  }
  degenerate <- !is.finite(b) || abs(b) > 20
  tibble(
    pc_beta = b, pc_se = se, pc_OR = exp(b),
    pc_CI_low = exp(ci[1]), pc_CI_high = exp(ci[2]),
    pc_P = p, number_phylo_axes = K, pc_method = method,
    pc_separation_used = sep_flag, pc_ci_type = ci_type,
    pc_ok = !degenerate,
    pc_warnings = paste(unique(warns), collapse = " | ")
  )
}

rows <- list()

for (test_id in names(pco_list)) {
  dd <- pco_list[[test_id]]$data
  trait <- if (grepl("disaccharide_use$", test_id)) "disaccharide_use" else
    if (grepl("hexose_alcohol_use$", test_id)) "hexose_alcohol_use" else NA
  if (is.na(trait)) next

  if (grepl("^me_", test_id)) {
    predictor <- "host_monocot"; coefs <- "host_monocot"; level <- "monocot_eudicot"
    ctr <- "host_monocot"
  } else if (grepl("^fam_", test_id)) {
    predictor <- "focal_family"; coefs <- "focal_family"; level <- "host_family"
    ctr <- unique(dd$host_family_clean[dd$focal_family == 1])
  } else if (grepl("^sp_", test_id)) {
    predictor <- "focal_host"; coefs <- "focal_host"; level <- "host_species"
    ctr <- unique(dd$host_species_clean[dd$focal_host == 1])
  } else if (grepl("^src_", test_id)) {
    predictor <- "focal_source"
    coefs <- "focal_source"
    level <- "broad_source"
    ctr <- if (grepl("^src_soil", test_id)) {
      "source_broadsoil/rhizosphere"
    } else "source_broadwater"
  } else next

  for (K in c(K_primary, K_sensitivity)) {
    for (j in seq_along(coefs)) {
      r <- fit_pc(dd, trait, predictor, coefs[j], K)
      if (is.null(r)) next
      r$metabolic_trait <- trait
      r$test_id <- test_id
      r$contrast <- ctr[j]
      r$ecological_level <- level
      r$n <- nrow(dd)
      rows[[paste(test_id, K, coefs[j])]] <- r
    }
  }
}

pc_all <- bind_rows(rows)
write_tsv(pc_all, file.path(cfg$paths$tables, "pc_adjusted_models.tsv"))
cat("PC/MDS-adjusted models written; n =", nrow(pc_all), "\n")
cat("non-degenerate significant (P<=0.05):\n")
print(pc_all %>% filter(pc_ok, pc_P <= 0.05) %>%
        select(metabolic_trait, contrast, number_phylo_axes, pc_OR,
               pc_CI_low, pc_CI_high, pc_P, pc_method), n = 40)
cat("\ndegenerate/NA rows:", sum(!pc_all$pc_ok | is.na(pc_all$pc_P)), "\n")


## 10. Preparation of phylogenetic covariance matrices

In [ ]:
# 10 - Build phylogenetic covariance (kinship) matrices from the analysis tree
suppressPackageStartupMessages({
  source(file.path("R", "utils.R"))
})

cfg <- read_config()
tree <- load_analysis_tree(cfg)
subsets <- readRDS(file.path(cfg$paths$models, "phyloglm_subsets.rds"))

# Full-tree covariance (correlation form: shared branch length proportion)
K_full <- ape::vcv.phylo(tree, corr = TRUE)
K_full <- as.matrix(K_full)
diag(K_full) <- 1

diag_report <- tibble(
  test_id = "full_tree",
  n = nrow(K_full),
  symmetric = isSymmetric(K_full),
  min_eigenvalue = min(eigen(K_full, symmetric = TRUE, only.values = TRUE)$values),
  min_diag = min(diag(K_full)),
  max_diag = max(diag(K_full))
)

kinship <- list(full = K_full)

# Per-test subset covariance matrices
for (test_id in names(subsets)) {
  pr <- prune_to_strains(tree, subsets[[test_id]])
  Ks <- as.matrix(ape::vcv.phylo(pr$tree, corr = TRUE))
  diag(Ks) <- 1
  kinship[[test_id]] <- Ks
  diag_report <- bind_rows(diag_report, tibble(
    test_id = test_id,
    n = nrow(Ks),
    symmetric = isSymmetric(Ks),
    min_eigenvalue = min(eigen(Ks, symmetric = TRUE, only.values = TRUE)$values),
    min_diag = min(diag(Ks)),
    max_diag = max(diag(Ks))
  ))
}

saveRDS(kinship, file.path(cfg$paths$pcs, "phylogenetic_covariance_matrices.rds"))
write_tsv(diag_report, file.path(cfg$paths$pcs, "phylogenetic_covariance_diagnostics.tsv"))

cat("Covariance matrices saved:", length(kinship), "\n")
print(diag_report, n = 45)


## 11. Kinship GLMM for disaccharide utilization

In [ ]:
args <- commandArgs(trailingOnly = TRUE)
if (length(args) < 1) stop("usage: Rscript glmm_rerun_disaccharide_single.R <test_id>")
test_id <- args[1]

run_dir <- "/public/home/luyt/Data/Ral/host_association/rscc_run"
out_dir <- "/public/home/luyt/Data/Ral/host_association/glmm_rerun"

setwd(run_dir)
suppressPackageStartupMessages({
  source(file.path("R", "utils.R"))
  source(file.path("R", "glmm.R"))
})

cfg <- read_config()
ck_dir <- file.path(out_dir, "checkpoints")
dir.create(ck_dir, recursive = TRUE, showWarnings = FALSE)
ck <- file.path(ck_dir, paste0(test_id, ".rds"))

spec <- data.frame(
  test_id = c(
    "src_soil_rhizosphere_disaccharide_use",
    "src_water_disaccharide_use",
    "me_disaccharide_use",
    "fam_Solanaceae_disaccharide_use",
    "fam_Musaceae_disaccharide_use",
    "fam_Zingiberaceae_disaccharide_use",
    "fam_Araceae_disaccharide_use",
    "fam_Fabaceae_disaccharide_use",
    "fam_Rosaceae_disaccharide_use",
    "fam_Cucurbitaceae_disaccharide_use",
    "sp_Solanum_tuberosum_disaccharide_use",
    "sp_Capsicum_annuum_disaccharide_use",
    "sp_Curcuma_spp__disaccharide_use",
    "sp_Musa_spp__disaccharide_use",
    "sp_Nicotiana_tabacum_disaccharide_use",
    "sp_Rosa_spp__disaccharide_use",
    "sp_Solanum_dulcamara_disaccharide_use",
    "sp_Solanum_lycopersicum_disaccharide_use",
    "sp_Solanum_melongena_disaccharide_use",
    "sp_Solanum_virginianum_disaccharide_use",
    "sp_Zingiber_officinale_disaccharide_use"
  ),
  trait = "disaccharide_use",
  predictor = c(
    "source_broad", "source_broad", "host_monocot",
    rep("focal_family", 7),
    rep("focal_host", 11)
  ),
  contrast = c(
    "source_broadsoil/rhizosphere",
    "source_broadwater",
    "host_monocot",
    "Solanaceae",
    "Musaceae",
    "Zingiberaceae",
    "Araceae",
    "Fabaceae",
    "Rosaceae",
    "Cucurbitaceae",
    "Solanum tuberosum",
    "Capsicum annuum",
    "Curcuma spp.",
    "Musa spp.",
    "Nicotiana tabacum",
    "Rosa spp.",
    "Solanum dulcamara",
    "Solanum lycopersicum",
    "Solanum melongena",
    "Solanum virginianum",
    "Zingiber officinale"
  ),
  stringsAsFactors = FALSE
)

sp <- spec[spec$test_id == test_id, ]
if (nrow(sp) != 1) stop("Unknown test_id: ", test_id)

subsets <- readRDS(file.path(cfg$paths$models, "phyloglm_subsets.rds"))
kinship <- readRDS(file.path(cfg$paths$pcs, "phylogenetic_covariance_matrices.rds"))

if (!test_id %in% names(subsets)) stop("Missing subset: ", test_id)
if (!test_id %in% names(kinship)) stop("Missing kinship: ", test_id)

dd <- subsets[[test_id]]
K <- kinship[[test_id]]
dd$strain <- rownames(dd)

if (sp$predictor == "source_broad") {
  if ("focal_source" %in% names(dd)) {
    dd$x <- as.integer(dd$focal_source)
  } else {
    dd$x <- as.integer(dd$source_broad == ifelse(test_id == "src_soil_rhizosphere_disaccharide_use", "soil/rhizosphere", "water"))
  }
} else {
  dd$x <- dd[[sp$predictor]]
}

level <- if (grepl("^me_", test_id)) {
  "monocot_eudicot"
} else if (grepl("^fam_", test_id)) {
  "host_family"
} else if (grepl("^sp_", test_id)) {
  "host_species"
} else {
  "broad_source"
}

cat("START", test_id, "n=", nrow(dd), "focal_n=", sum(dd$x == 1, na.rm = TRUE), "\n")
res <- fit_one_glmm(
  dd, K, "disaccharide_use", cfg$settings$seed,
  iter = 2000, warmup = 1000, chains = 10, cores = 10
)
row <- extract_glmm_row(res, "disaccharide_use", sp$contrast, level, test_id)
saveRDS(row, ck)

cat(
  "DONE", test_id,
  "ok=", isTRUE(row$glmm_ok),
  "OR=", ifelse(is.na(row$glmm_OR), "NA", row$glmm_OR),
  "Pr_beta_gt0=", ifelse(is.na(row$glmm_posterior_prob_beta_gt0), "NA", row$glmm_posterior_prob_beta_gt0),
  "Rhat=", ifelse(is.na(row$glmm_Rhat), "NA", row$glmm_Rhat),
  "Bulk_ESS=", ifelse(is.na(row$glmm_Bulk_ESS), "NA", row$glmm_Bulk_ESS),
  "divergences=", ifelse(is.na(row$glmm_divergences), "NA", row$glmm_divergences),
  "\n"
)


## 12. Experimental-phenotype-only phyloglm sensitivity analysis

In [ ]:
# 17 - Experimental-only phenotype sensitivity analysis
# Uses only strains whose phenotype was experimentally determined.
# Provides phyloglm + PC/MDS-adjusted (K = 5) sensitivity for every test.
suppressPackageStartupMessages({
  source(file.path("R", "utils.R"))
})

cfg <- read_config()
set.seed(cfg$settings$seed)
dat <- load_curated_metadata(cfg)
tree <- load_analysis_tree(cfg)

dat <- dat %>%
  mutate(
    disaccharide_use = as.integer(disaccharide_use),
    hexose_alcohol_use = as.integer(hexose_alcohol_use),
    host_monocot = case_when(
      host_major_clade == "monocot" ~ 1L,
      host_major_clade == "eudicot" ~ 0L,
      TRUE ~ NA_integer_
    )
  )

exp_d <- dat %>% filter(disaccharide_source == "experimental")
exp_h <- dat %>% filter(hexose_alcohol_source == "experimental")
rows <- list()

fit_exp <- function(trait, dd, test_label, fdr_group, focal_col,
                    contrast_label = focal_col) {
  pr <- prune_to_strains(tree, dd)
  sep <- FALSE
  foc <- dd[[focal_col]] == 1
  if (sum(foc) >= 2 && length(unique(dd[[trait]][foc])) < 2) sep <- TRUE
  fit <- fit_phyloglm(as.formula(paste(trait, "~", focal_col)),
                      pr$data, pr$tree)
  r <- extract_phyloglm_row(fit, focal_col, trait, test_label)
  if (sep) {
    r <- r %>% mutate(
      phyloglm_OR = NA_real_, phyloglm_P = NA_real_, phyloglm_alpha = NA_real_,
      phyloglm_error = "complete_separation_focal_trait_invariant"
      ,
      phyloglm_warnings = NA_character_
    )
  }
  # PC/MDS-adjusted (K = 5, from patristic distances)
  D <- ape::cophenetic.phylo(pr$tree)
  pco <- cmdscale(D, k = 5, eig = TRUE)
  colnames(pco$points) <- paste0("PhyloPC", 1:5)
  ddp <- cbind(pr$data, pco$points)
  pc_fml <- as.formula(paste(trait, "~", focal_col,
                             "+ PhyloPC1 + PhyloPC2 + PhyloPC3 + PhyloPC4 + PhyloPC5"))
  g <- tryCatch(glm(pc_fml, data = ddp, family = binomial()),
                error = function(e) NULL)
  if (!is.null(g) && !detect_separation(g)) {
    co <- summary(g)$coefficients[focal_col, ]
    r$pc_OR <- exp(co["Estimate"])
    r$pc_CI_low <- exp(co["Estimate"] - qnorm(0.975) * co["Std. Error"])
    r$pc_CI_high <- exp(co["Estimate"] + qnorm(0.975) * co["Std. Error"])
    r$pc_P <- co["Pr(>|z|)"]
    r$pc_method <- "glm"
  } else {
    lf <- tryCatch(
      logistf::logistf(pc_fml, data = ddp, pl = TRUE,
                       control = logistf::logistf.control(maxit = 1000, maxstep = 20)),
      error = function(e) NULL)
    if (!is.null(lf) && focal_col %in% names(lf$coefficients)) {
      ci <- tryCatch(confint(lf), error = function(e) matrix(NA, 2, 2))
      b <- lf$coefficients[focal_col]; p <- lf$prob[focal_col]
      se <- sqrt(diag(lf$var))[focal_col]
      ci_row <- if (focal_col %in% rownames(ci)) ci[focal_col, ] else
        c(b - qnorm(0.975) * se, b + qnorm(0.975) * se)
      r$pc_OR <- exp(b)
      r$pc_CI_low <- exp(ci_row[1]); r$pc_CI_high <- exp(ci_row[2])
      r$pc_P <- p
      r$pc_method <- "logistf"
    }
  }
  if (is.null(r$pc_OR)) {
    r$pc_OR <- NA_real_; r$pc_CI_low <- NA_real_; r$pc_CI_high <- NA_real_
    r$pc_P <- NA_real_; r$pc_method <- NA_character_
  }
  r$fdr_group <- fdr_group
  r$n <- fit$n
  r$test_label <- test_label
  r$contrast_label <- contrast_label
  r
}

# broad source: binary one-vs-rest contrasts (stable formulation)
src_bin <- list(
  list(level = "soil/rhizosphere", label = "source_broadsoil/rhizosphere"),
  list(level = "water", label = "source_broadwater"))
for (pair in list(c("disaccharide_use", "exp_d"), c("hexose_alcohol_use", "exp_h"))) {
  tr <- pair[1]; base <- get(pair[2]) %>%
    filter(source_broad %in% c("plant", "soil/rhizosphere", "water"))
  for (sb in src_bin) {
    dd <- base %>% mutate(focal_source = as.integer(source_broad == sb$level))
    r <- fit_exp(tr, dd, paste0("exp_src_", tr, "_", sb$label),
                 paste0("exp_broad_source_", tr),
                 focal_col = "focal_source", contrast_label = sb$label)
    rows[[paste(tr, sb$label, "src")]] <- r
  }
}

# monocot vs eudicot
for (pair in list(c("disaccharide_use", "exp_d"), c("hexose_alcohol_use", "exp_h"))) {
  tr <- pair[1]; dd <- get(pair[2]) %>% filter(!is.na(host_monocot))
  rows[[paste(tr, "me")]] <- fit_exp(tr, dd, paste0("exp_me_", tr),
                                     paste0("exp_monocot_eudicot_", tr),
                                     focal_col = "host_monocot")
}

# families (n >= 5 in experimental subset)
for (pair in list(c("disaccharide_use", "exp_d"), c("hexose_alcohol_use", "exp_h"))) {
  tr <- pair[1]; base <- get(pair[2])
  fams <- base %>% filter(source_broad == "plant", !is.na(host_family_clean)) %>%
    count(host_family_clean, name = "n") %>% filter(n >= 5)
  for (fm in fams$host_family_clean) {
    dd <- base %>% filter(source_broad == "plant", !is.na(host_family_clean)) %>%
      mutate(focal_family = as.integer(host_family_clean == fm))
    r <- fit_exp(tr, dd, paste0("exp_family_", tr, "_", fm),
                 paste0("exp_family_", tr),
                 focal_col = "focal_family", contrast_label = fm)
    rows[[paste(tr, "fam", fm)]] <- r
  }
}

# top species (n >= 3 in experimental subset)
top_sp <- dat %>% filter(source_broad == "plant", !is.na(host_species_clean),
                         host_species_clean != "unknown") %>%
  count(host_species_clean, name = "n") %>% arrange(desc(n)) %>%
  filter(n >= cfg$settings$min_host_species_n)

for (pair in list(c("disaccharide_use", "exp_d"), c("hexose_alcohol_use", "exp_h"))) {
  tr <- pair[1]; base <- get(pair[2])
  for (sp in top_sp$host_species_clean) {
    dd <- base %>% filter(source_broad == "plant", !is.na(host_species_clean),
                          host_species_clean != "unknown") %>%
      mutate(focal_host = as.integer(host_species_clean == sp))
    if (sum(dd$focal_host) < 3 || nrow(dd) - sum(dd$focal_host) < 3) next
    r <- fit_exp(tr, dd, paste0("exp_species_", tr, "_", sp),
                 paste0("exp_species_", tr),
                 focal_col = "focal_host", contrast_label = sp)
    rows[[paste(tr, "sp", sp)]] <- r
  }
}

out <- bind_rows(rows)
out <- add_fdr(out, "phyloglm_P", "fdr_group")
write_tsv(out, file.path(cfg$paths$tables, "experimental_only_sensitivity.tsv"))

cat("Experimental-only sensitivity written; n =", nrow(out), "\n")
print(out %>% filter(phyloglm_P <= 0.1 | pc_P <= 0.1) %>%
        select(metabolic_trait, test_label, n, phyloglm_OR, phyloglm_P,
               pc_OR, pc_P, fdr), n = 30)


## 13. Geography-adjusted phyloglm sensitivity analysis

In [ ]:
suppressPackageStartupMessages({
  library(readr)
  library(dplyr)
  library(tibble)
  library(ape)
  library(phylolm)
})

myresult_dir <- "E:/Data/Ralstonia/Pycharm/graduation2/new_analysis/association/myresult"
metadata_path <- file.path(myresult_dir, "tables", "curated_metadata.tsv")
tree_path <- "E:/Data/Ralstonia/Pycharm/graduation2/new_analysis/SHAP/teacher_rerun_batch2_n10/results/models/analysis_tree.nwk"
out_dir <- file.path(myresult_dir, "rerun")
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

dat <- read_tsv(metadata_path, col_types = cols(.default = col_character())) %>%
  mutate(
    disaccharide_use = as.integer(disaccharide_use),
    host_monocot = case_when(
      host_major_clade == "monocot" ~ 1L,
      host_major_clade == "eudicot" ~ 0L,
      TRUE ~ NA_integer_
    ),
    continent = if_else(is.na(Continent) | Continent == "Unknown" | Continent == "",
                        "unknown", Continent)
  )

tree <- ape::read.tree(tree_path)

prune_to_strains_local <- function(tree, data) {
  dd <- data %>%
    mutate(strain_id = trimws(as.character(strain_id))) %>%
    filter(!is.na(strain_id), strain_id %in% tree$tip.label) %>%
    distinct(strain_id, .keep_all = TRUE)
  tr <- ape::keep.tip(tree, dd$strain_id)
  dd <- as.data.frame(dd[match(tr$tip.label, dd$strain_id), , drop = FALSE])
  rownames(dd) <- dd$strain_id
  list(data = dd, tree = tr)
}

fit_phyloglm_local <- function(formula, data, tree) {
  warn <- character()
  fit <- tryCatch(
    withCallingHandlers(
      phylolm::phyloglm(formula, data = data, phy = tree,
                        method = "logistic_MPLE", boot = 0),
      warning = function(w) {
        warn <<- c(warn, conditionMessage(w))
        invokeRestart("muffleWarning")
      }
    ),
    error = function(e) e
  )
  warn_msg <- paste(unique(warn), collapse = " | ")
  warn_msg <- gsub("[\\r\\n\\t]+", " ", warn_msg, perl = TRUE)
  if (!nzchar(trimws(warn_msg))) warn_msg <- NA_character_
  if (inherits(fit, "error")) {
    return(list(ok = FALSE, n = NA_integer_, coefs = NULL,
                error = conditionMessage(fit), warnings = warn_msg))
  }
  list(ok = TRUE, n = fit$n,
       coefs = tryCatch(summary(fit)$coefficients, error = function(e) NULL),
       error = NA_character_, warnings = warn_msg)
}

extract_coef <- function(fit, pred) {
  if (is.null(fit) || !isTRUE(fit$ok) || is.null(fit$coefs) || !(pred %in% rownames(fit$coefs))) {
    return(tibble(beta = NA_real_, se = NA_real_, P = NA_real_,
                  OR = NA_real_, CI_low = NA_real_, CI_high = NA_real_))
  }
  co <- fit$coefs
  pcol <- intersect(c("p.value", "Pr(>|z|)", "Pr(>z)"), colnames(co))[1]
  b <- unname(co[pred, "Estimate"])
  se <- unname(co[pred, "StdErr"])
  tibble(
    beta = b,
    se = se,
    P = if (!is.na(pcol)) unname(co[pred, pcol]) else NA_real_,
    OR = exp(b),
    CI_low = exp(b - qnorm(0.975) * se),
    CI_high = exp(b + qnorm(0.975) * se)
  )
}

tests <- tribble(
  ~level, ~label, ~test_id, ~kind,
  "broad source", "Soil/rhizosphere", "src_soil_rhizosphere_disaccharide_use", "soil",
  "broad source", "Water", "src_water_disaccharide_use", "water",
  "plant cotyledon", "Monocotyledon", "me_disaccharide_use", "monocot",
  "family", "Solanaceae", "fam_Solanaceae_disaccharide_use", "family",
  "family", "Musaceae", "fam_Musaceae_disaccharide_use", "family",
  "family", "Zingiberaceae", "fam_Zingiberaceae_disaccharide_use", "family",
  "family", "Araceae", "fam_Araceae_disaccharide_use", "family",
  "family", "Fabaceae", "fam_Fabaceae_disaccharide_use", "family",
  "family", "Rosaceae", "fam_Rosaceae_disaccharide_use", "family",
  "family", "Cucurbitaceae", "fam_Cucurbitaceae_disaccharide_use", "family",
  "species", "Solanum tuberosum", "sp_Solanum_tuberosum_disaccharide_use", "species",
  "species", "Capsicum annuum", "sp_Capsicum_annuum_disaccharide_use", "species",
  "species", "Curcuma spp.", "sp_Curcuma_spp__disaccharide_use", "species",
  "species", "Musa spp.", "sp_Musa_spp__disaccharide_use", "species",
  "species", "Nicotiana tabacum", "sp_Nicotiana_tabacum_disaccharide_use", "species",
  "species", "Rosa spp.", "sp_Rosa_spp__disaccharide_use", "species",
  "species", "Solanum dulcamara", "sp_Solanum_dulcamara_disaccharide_use", "species",
  "species", "Solanum lycopersicum", "sp_Solanum_lycopersicum_disaccharide_use", "species",
  "species", "Solanum melongena", "sp_Solanum_melongena_disaccharide_use", "species",
  "species", "Solanum virginianum", "sp_Solanum_virginianum_disaccharide_use", "species",
  "species", "Zingiber officinale", "sp_Zingiber_officinale_disaccharide_use", "species"
)

rows <- vector("list", nrow(tests))

for (i in seq_len(nrow(tests))) {
  tt <- tests[i, ]
  dd <- dat %>% filter(!is.na(disaccharide_use), continent != "unknown")
  pred <- NA_character_

  if (tt$kind == "soil") {
    dd <- dd %>%
      filter(source_broad %in% c("plant", "soil/rhizosphere", "water")) %>%
      mutate(focal_source = as.integer(source_broad == "soil/rhizosphere"))
    pred <- "focal_source"
  } else if (tt$kind == "water") {
    dd <- dd %>%
      filter(source_broad %in% c("plant", "soil/rhizosphere", "water")) %>%
      mutate(focal_source = as.integer(source_broad == "water"))
    pred <- "focal_source"
  } else if (tt$kind == "monocot") {
    dd <- dd %>% filter(!is.na(host_monocot))
    pred <- "host_monocot"
  } else if (tt$kind == "family") {
    dd <- dd %>%
      filter(source_broad == "plant", !is.na(host_family_clean), host_family_clean != "") %>%
      mutate(focal_family = as.integer(host_family_clean == tt$label))
    pred <- "focal_family"
  } else if (tt$kind == "species") {
    dd <- dd %>%
      filter(source_broad == "plant", !is.na(host_species_clean),
             host_species_clean != "", host_species_clean != "unknown") %>%
      mutate(focal_host = as.integer(host_species_clean == tt$label))
    pred <- "focal_host"
  }

  pr <- prune_to_strains_local(tree, dd)
  focal <- pr$data[[pred]]
  focal_n <- sum(focal == 1, na.rm = TRUE)
  focal_prevalence <- if (focal_n > 0) mean(pr$data$disaccharide_use[focal == 1], na.rm = TRUE) else NA_real_
  separation <- !is.na(focal_prevalence) && focal_prevalence %in% c(0, 1)

  fit_geo <- fit_phyloglm_local(as.formula(paste("disaccharide_use ~", pred, "+ continent")),
                                pr$data, pr$tree)
  fit_nogeo <- fit_phyloglm_local(as.formula(paste("disaccharide_use ~", pred)),
                                  pr$data, pr$tree)

  cgeo <- extract_coef(fit_geo, pred)
  cnogeo <- extract_coef(fit_nogeo, pred)

  continent_rows <- if (!is.null(fit_geo$coefs)) grepl("^continent", rownames(fit_geo$coefs)) else logical()
  continent_effect_any_sig <- if (!is.null(fit_geo$coefs) && any(continent_rows)) {
    any(fit_geo$coefs[continent_rows, "p.value"] < 0.05, na.rm = TRUE)
  } else {
    NA
  }

  rows[[i]] <- tibble(
    level = tt$level,
    label = tt$label,
    test_id = tt$test_id,
    metabolic_trait = "disaccharide_use",
    n = fit_geo$n,
    focal_n = focal_n,
    focal_prevalence = focal_prevalence,
    separation = separation,
    beta_no_geo = cnogeo$beta,
    P_no_geo = cnogeo$P,
    OR_no_geo = cnogeo$OR,
    beta_with_continent = cgeo$beta,
    P_with_continent = cgeo$P,
    OR_with_continent = cgeo$OR,
    OR_CI_low = cgeo$CI_low,
    OR_CI_high = cgeo$CI_high,
    continent_effect_any_sig = continent_effect_any_sig,
    fit_geo_ok = fit_geo$ok,
    fit_geo_error = fit_geo$error,
    fit_geo_warnings = fit_geo$warnings,
    fit_nogeo_ok = fit_nogeo$ok,
    fit_nogeo_error = fit_nogeo$error,
    fit_nogeo_warnings = fit_nogeo$warnings
  )
}

out <- bind_rows(rows) %>%
  group_by(level) %>%
  mutate(
    fdr_with_continent = p.adjust(P_with_continent, method = "BH"),
    fdr_no_geo = p.adjust(P_no_geo, method = "BH")
  ) %>%
  ungroup()

out_with_dicot <- bind_rows(
  out,
  out %>%
    filter(level == "plant cotyledon", label == "Monocotyledon") %>%
    mutate(
      label = "Dicotyledon",
      OR_no_geo = 1 / OR_no_geo,
      OR_with_continent = 1 / OR_with_continent,
      OR_CI_low = 1 / OR_CI_high,
      OR_CI_high = 1 / OR_CI_low,
      beta_no_geo = -beta_no_geo,
      beta_with_continent = -beta_with_continent
    )
)

write_tsv(out, file.path(out_dir, "geography_sensitivity_disaccharide_full_model_rows.tsv"))
write_tsv(out_with_dicot, file.path(out_dir, "geography_sensitivity_disaccharide_full_with_dicotyledon.tsv"))

cat("Wrote:\n")
cat(file.path(out_dir, "geography_sensitivity_disaccharide_full_model_rows.tsv"), "\n")
cat(file.path(out_dir, "geography_sensitivity_disaccharide_full_with_dicotyledon.tsv"), "\n")
print(out_with_dicot %>% select(level, label, test_id, n, focal_n, focal_prevalence, separation,
                                OR_with_continent, P_with_continent, fdr_with_continent,
                                continent_effect_any_sig, fit_geo_ok, fit_geo_error),
      n = Inf)


## 14. Within-cotyledon phyloglm sensitivity analysis

In [ ]:
suppressPackageStartupMessages({
  library(readr)
  library(dplyr)
  library(tibble)
  library(purrr)
  library(ape)
  library(phylolm)
})

base_dir <- "E:/Data/Ralstonia/Pycharm/graduation2/new_analysis/association/myresult"
out_dir <- file.path(base_dir, "rerun")
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

source(file.path(base_dir, "code", "utils.R"))

dat <- readr::read_tsv(
  file.path(base_dir, "tables", "curated_metadata.tsv"),
  col_types = cols(.default = col_character())
) %>%
  mutate(
    disaccharide_use = as.integer(disaccharide_use),
    hexose_alcohol_use = as.integer(hexose_alcohol_use)
  )

tree <- ape::read.tree(
  "E:/Data/Ralstonia/Pycharm/graduation2/new_analysis/SHAP/analysis_tree_teacher_processed.nwk"
)
if (is.null(tree$edge.length)) {
  stop("analysis_tree.nwk has no branch lengths")
}

traits <- c("disaccharide_use", "hexose_alcohol_use")
plant <- dat %>%
  filter(source_broad == "plant",
         host_major_clade %in% c("monocot", "eudicot"))

mark_complete_separation <- function(row) {
  row %>% mutate(
    phyloglm_beta = NA_real_,
    phyloglm_se = NA_real_,
    phyloglm_OR = NA_real_,
    phyloglm_CI_low = NA_real_,
    phyloglm_CI_high = NA_real_,
    phyloglm_P = NA_real_,
    phyloglm_alpha = NA_real_,
    phyloglm_error = "complete_separation_focal_trait_invariant"
  )
}

fit_same_major <- function(trait, data, focal_col, test_label, fdr_group) {
  pr <- prune_to_strains(tree, data)
  dd <- pr$data
  tr <- pr$tree

  focal <- dd[[focal_col]] == 1
  focal_values <- dd[[trait]][focal]
  focal_prevalence <- mean(focal_values, na.rm = TRUE)
  focal_n <- sum(focal, na.rm = TRUE)
  ref_n <- nrow(dd) - focal_n
  separation <- length(unique(focal_values[!is.na(focal_values)])) < 2

  if (isTRUE(separation)) {
    row <- tibble(
      metabolic_trait = trait,
      ecological_level = test_label,
      predictor = focal_col,
      n = nrow(dd),
      phyloglm_beta = NA_real_,
      phyloglm_se = NA_real_,
      phyloglm_OR = NA_real_,
      phyloglm_CI_low = NA_real_,
      phyloglm_CI_high = NA_real_,
      phyloglm_P = NA_real_,
      phyloglm_alpha = NA_real_,
      phyloglm_converged = NA,
      phyloglm_error = "complete_separation_focal_trait_invariant",
      phyloglm_warnings = NA_character_
    )
  } else {
    fit <- fit_phyloglm(as.formula(paste(trait, "~", focal_col)), dd, tr)
    row <- extract_phyloglm_row(fit, focal_col, trait, test_label)
    row$n <- fit$n
  }

  row$fdr_group <- fdr_group
  row$focal_n <- focal_n
  row$ref_n <- ref_n
  row$focal_prevalence <- focal_prevalence
  row$separation <- separation
  row$sparse <- focal_n < 5 || ref_n < 5
  row
}

family_rows <- list()
family_counts <- plant %>%
  filter(!is.na(host_family_clean)) %>%
  count(host_family_clean, name = "focal_n") %>%
  filter(focal_n >= 10)

for (trt in traits) {
  for (fam in family_counts$host_family_clean) {
    clade <- plant$host_major_clade[plant$host_family_clean == fam][1]
    dd <- plant %>%
      filter(host_major_clade == clade, !is.na(host_family_clean)) %>%
      mutate(focal_family = as.integer(host_family_clean == fam))
    if (sum(dd$focal_family, na.rm = TRUE) < 5 ||
        nrow(dd) - sum(dd$focal_family, na.rm = TRUE) < 5) next
    row <- fit_same_major(
      trt, dd, "focal_family",
      paste0("family_same_major_cotyledon_", fam),
      paste0("family_same_major_cotyledon_", trt)
    )
    row$focal_family <- fam
    row$clade <- clade
    family_rows[[paste(trt, fam, sep = "||")]] <- row
  }
}

family_out <- bind_rows(family_rows) %>%
  group_by(fdr_group) %>%
  mutate(fdr = p.adjust(phyloglm_P, method = "BH")) %>%
  ungroup()

species_rows <- list()
species_counts <- plant %>%
  filter(!is.na(host_species_clean), host_species_clean != "unknown") %>%
  count(host_species_clean, name = "focal_n") %>%
  filter(focal_n >= 10)

for (trt in traits) {
  for (sp in species_counts$host_species_clean) {
    clade <- plant$host_major_clade[plant$host_species_clean == sp][1]
    dd <- plant %>%
      filter(host_major_clade == clade,
             !is.na(host_species_clean),
             host_species_clean != "unknown") %>%
      mutate(focal_host = as.integer(host_species_clean == sp))
    if (sum(dd$focal_host, na.rm = TRUE) < 5 ||
        nrow(dd) - sum(dd$focal_host, na.rm = TRUE) < 5) next
    row <- fit_same_major(
      trt, dd, "focal_host",
      paste0("species_same_major_cotyledon_", sp),
      paste0("species_same_major_cotyledon_", trt)
    )
    row$host_species <- sp
    row$clade <- clade
    species_rows[[paste(trt, sp, sep = "||")]] <- row
  }
}

species_out <- bind_rows(species_rows) %>%
  group_by(fdr_group) %>%
  mutate(fdr = p.adjust(phyloglm_P, method = "BH")) %>%
  ungroup()

readr::write_tsv(
  family_out,
  file.path(out_dir, "same_major_cotyledon_family_no_complete_separation.tsv")
)
readr::write_tsv(
  species_out,
  file.path(out_dir, "same_major_cotyledon_species_no_complete_separation.tsv")
)

combined <- bind_rows(
  family_out %>%
    transmute(
      level = "family",
      label = focal_family,
      metabolic_trait,
      clade,
      n,
      focal_n,
      ref_n,
      focal_prevalence,
      phyloglm_OR,
      phyloglm_CI_low,
      phyloglm_CI_high,
      phyloglm_P,
      fdr,
      separation,
      phyloglm_error
    ),
  species_out %>%
    transmute(
      level = "species",
      label = host_species,
      metabolic_trait,
      clade,
      n,
      focal_n,
      ref_n,
      focal_prevalence,
      phyloglm_OR,
      phyloglm_CI_low,
      phyloglm_CI_high,
      phyloglm_P,
      fdr,
      separation,
      phyloglm_error
    )
)

readr::write_tsv(
  combined,
  file.path(out_dir, "same_major_cotyledon_no_complete_separation_combined.tsv")
)

cat("same-major-cotyledon rerun complete\n")
cat("family rows:", nrow(family_out), "\n")
cat("species rows:", nrow(species_out), "\n")
cat("complete separation rows:\n")
print(combined %>%
        filter(separation) %>%
        select(level, label, metabolic_trait, focal_n, focal_prevalence,
               phyloglm_error),
      n = 50)


## 15. Pagel correlated-evolution tests using the full reference sets

In [ ]:
args <- commandArgs(trailingOnly = TRUE)
if (length(args) < 1) stop("usage: Rscript corr_evo_vsall_bootall.R <pair_label>")
target_pair <- args[1]

suppressPackageStartupMessages({
  source(file.path("R", "utils.R"))
  library(dplyr)
  library(readr)
  library(tibble)
  library(phytools)
  library(parallel)
})

cfg <- read_config()
set.seed(cfg$settings$seed)
dat <- load_curated_metadata(cfg)
tree <- load_analysis_tree(cfg)

out_pair_dir <- file.path(cfg$paths$evolutionary, "pair_results")
dir.create(out_pair_dir, recursive = TRUE, showWarnings = FALSE)

dat <- dat %>%
  mutate(
    disaccharide_use = as.integer(disaccharide_use),
    hexose_alcohol_use = as.integer(hexose_alcohol_use),
    source_broad = as.character(source_broad),
    host_major_clade = as.character(host_major_clade),
    host_family_clean = as.character(host_family_clean),
    host_species_clean = as.character(host_species_clean)
  )

source_dat <- dat %>% filter(source_broad %in% c("plant", "soil/rhizosphere", "water"))
plant <- dat %>% filter(source_broad == "plant", !is.na(host_family_clean), !is.na(host_major_clade))
plant_host <- plant %>% filter(!is.na(host_species_clean), host_species_clean != "unknown")

make_pair <- function(dd, eco_var, meta_var, label, level, focal_label, reference) {
  dd <- dd %>% filter(!is.na(.data[[eco_var]]), !is.na(.data[[meta_var]]))
  pr <- prune_to_strains(tree, dd)
  x <- factor(pr$data[[eco_var]])
  y <- factor(pr$data[[meta_var]])
  names(x) <- names(y) <- rownames(pr$data)
  list(x = x, y = y, tree = pr$tree, label = label,
       level = level, focal_label = focal_label, reference = reference)
}

pairs <- list()
add_pair <- function(label, dd, level, focal_label, reference) {
  pairs[[label]] <<- make_pair(dd, "foc", "disaccharide_use", label, level, focal_label, reference)
}

add_pair("Soil_rhizosphere_vs_all_sources_x_disaccharide",
         source_dat %>% mutate(foc = as.integer(source_broad == "soil/rhizosphere")),
         "broad source", "Soil/rhizosphere", "all other sources")
add_pair("Water_vs_all_sources_x_disaccharide",
         source_dat %>% mutate(foc = as.integer(source_broad == "water")),
         "broad source", "Water", "all other sources")
add_pair("Dicotyledon_vs_allplants_x_disaccharide",
         plant %>% mutate(foc = as.integer(host_major_clade == "eudicot")),
         "plant cotyledon", "Dicotyledon", "all other plant hosts")
add_pair("Monocotyledon_vs_allplants_x_disaccharide",
         plant %>% mutate(foc = as.integer(host_major_clade == "monocot")),
         "plant cotyledon", "Monocotyledon", "all other plant hosts")

for (fam in c("Solanaceae", "Musaceae", "Zingiberaceae", "Araceae", "Fabaceae", "Rosaceae", "Cucurbitaceae")) {
  label <- paste0(gsub("[^A-Za-z0-9]+", "_", fam), "_vs_allplants_x_disaccharide")
  add_pair(label, plant %>% mutate(foc = as.integer(host_family_clean == fam)),
           "family", fam, "all other plant hosts")
}


# Family within-clade comparisons: focal family vs other families within the same plant major clade.
for (fam in c("Musaceae", "Araceae", "Zingiberaceae")) {
  label <- paste0(gsub("[^A-Za-z0-9]+", "_", fam), "_vs_monocots_x_disaccharide")
  add_pair(label,
           plant %>% filter(host_major_clade == "monocot") %>% mutate(foc = as.integer(host_family_clean == fam)),
           "family within clade", fam, "other monocot plant hosts")
}
for (fam in c("Solanaceae", "Fabaceae", "Rosaceae", "Cucurbitaceae")) {
  label <- paste0(gsub("[^A-Za-z0-9]+", "_", fam), "_vs_eudicots_x_disaccharide")
  add_pair(label,
           plant %>% filter(host_major_clade == "eudicot") %>% mutate(foc = as.integer(host_family_clean == fam)),
           "family within clade", fam, "other eudicot plant hosts")
}
species_map <- c(
  "Solanum_tuberosum" = "Solanum tuberosum",
  "Capsicum_annuum" = "Capsicum annuum",
  "Curcuma_spp" = "Curcuma spp.",
  "Musa_spp" = "Musa spp.",
  "Nicotiana_tabacum" = "Nicotiana tabacum",
  "Rosa_spp" = "Rosa spp.",
  "Solanum_dulcamara" = "Solanum dulcamara",
  "Solanum_lycopersicum" = "Solanum lycopersicum",
  "Solanum_melongena" = "Solanum melongena",
  "Solanum_virginianum" = "Solanum virginianum",
  "Zingiber_officinale" = "Zingiber officinale"
)
for (nm in names(species_map)) {
  sp <- unname(species_map[[nm]])
  label <- paste0(nm, "_vs_allplants_x_disaccharide")
  add_pair(label, plant_host %>% mutate(foc = as.integer(host_species_clean == sp)),
           "species", sp, "all other plant hosts")
}

if (!target_pair %in% names(pairs)) {
  stop("Unknown pair: ", target_pair, "\nAvailable: ", paste(names(pairs), collapse = ", "))
}
p <- pairs[[target_pair]]

build_Q <- function(fit, state_levels = NULL) {
  k <- nrow(fit$index.matrix)
  Q <- matrix(0, k, k)
  idx <- fit$index.matrix
  ii <- which(idx > 0, arr.ind = TRUE)
  Q[ii] <- fit$rates[idx[ii]]
  diag(Q) <- -rowSums(Q)
  dimnames(Q) <- dimnames(idx)
  if (is.null(rownames(Q)) && !is.null(state_levels) && length(state_levels) == k) {
    rownames(Q) <- colnames(Q) <- state_levels
  }
  Q
}

recode_sim_states <- function(states, state_levels) {
  out <- as.character(states)
  if (all(out %in% state_levels)) {
    return(factor(out, levels = state_levels))
  }
  idx <- suppressWarnings(as.integer(out))
  use_idx <- !is.na(idx) & idx >= 1 & idx <= length(state_levels)
  out[use_idx] <- state_levels[idx[use_idx]]
  factor(out, levels = state_levels)
}

extract_rates <- function(Q, pair) {
  st <- rownames(Q)
  if (!all(c("0|0", "0|1", "1|0", "1|1") %in% st)) {
    return(tibble(pair = pair,
      q_gain_focal = NA_real_, q_gain_reference = NA_real_,
      q_loss_focal = NA_real_, q_loss_reference = NA_real_,
      q_habitat_gain_trait_pos = NA_real_, q_habitat_gain_trait_neg = NA_real_,
      q_habitat_loss_trait_pos = NA_real_, q_habitat_loss_trait_neg = NA_real_,
      metabolic_gain_rate_ratio = NA_real_, metabolic_loss_rate_ratio = NA_real_,
      ecological_transition_rate_ratio = NA_real_))
  }
  get <- function(i, j) Q[i, j]
  q_gain_focal <- get("1|0", "1|1")
  q_gain_reference <- get("0|0", "0|1")
  q_loss_focal <- get("1|1", "1|0")
  q_loss_reference <- get("0|1", "0|0")
  q_hab_gain_pos <- get("0|1", "1|1")
  q_hab_gain_neg <- get("0|0", "1|0")
  q_hab_loss_pos <- get("1|1", "0|1")
  q_hab_loss_neg <- get("1|0", "0|0")
  tibble(pair = pair,
    q_gain_focal = q_gain_focal, q_gain_reference = q_gain_reference,
    q_loss_focal = q_loss_focal, q_loss_reference = q_loss_reference,
    q_habitat_gain_trait_pos = q_hab_gain_pos,
    q_habitat_gain_trait_neg = q_hab_gain_neg,
    q_habitat_loss_trait_pos = q_hab_loss_pos,
    q_habitat_loss_trait_neg = q_hab_loss_neg,
    metabolic_gain_rate_ratio = q_gain_focal / (q_gain_reference + 1e-12),
    metabolic_loss_rate_ratio = q_loss_focal / (q_loss_reference + 1e-12),
    ecological_transition_rate_ratio = q_hab_gain_pos / (q_hab_gain_neg + 1e-12))
}

safe <- gsub("[^A-Za-z0-9_]", "_", target_pair)
row_path <- file.path(out_pair_dir, paste0(safe, ".row.rds"))
rate_path <- file.path(out_pair_dir, paste0(safe, ".rates.rds"))
fit_path <- file.path(out_pair_dir, paste0(safe, ".fits.rds"))

cat("START", target_pair, "n=", length(p$x), "focal_n=", sum(p$x == 1), "\n")

fx <- tryCatch(phytools::fitMk(p$tree, p$x, model = "ER"), error = function(e) e)
fy <- tryCatch(phytools::fitMk(p$tree, p$y, model = "ER"), error = function(e) e)
if (inherits(fx, "error") || inherits(fy, "error")) {
  row <- tibble(pair = target_pair, level = p$level, label = p$focal_label, reference = p$reference,
                n = length(p$x), focal_n = sum(p$x == 1),
                logLik_independent = NA_real_, logLik_dependent = NA_real_,
                AIC_independent = NA_real_, AIC_dependent = NA_real_,
                delta_AIC = NA_real_, likelihood_ratio = NA_real_, df = 10, P = NA_real_, fit_ok = FALSE,
                bootstrap_P = NA_real_, bootstrap_nsim = NA_real_, bootstrap_n_valid = NA_real_)
  saveRDS(row, row_path); write_tsv(row, file.path(out_pair_dir, paste0(safe, ".row.tsv")))
  stop("Independent model failed for ", target_pair)
}

z <- factor(paste0(p$x, "|", p$y))
names(z) <- names(p$x)
fz <- tryCatch(phytools::fitMk(p$tree, z, model = "ARD"), error = function(e) e)
if (inherits(fz, "error")) {
  row <- tibble(pair = target_pair, level = p$level, label = p$focal_label, reference = p$reference,
                n = length(p$x), focal_n = sum(p$x == 1),
                logLik_independent = fx$logLik + fy$logLik, logLik_dependent = NA_real_,
                AIC_independent = NA_real_, AIC_dependent = NA_real_, delta_AIC = NA_real_,
                likelihood_ratio = NA_real_, df = 10, P = NA_real_, fit_ok = FALSE,
                bootstrap_P = NA_real_, bootstrap_nsim = NA_real_, bootstrap_n_valid = NA_real_)
  saveRDS(row, row_path); write_tsv(row, file.path(out_pair_dir, paste0(safe, ".row.tsv")))
  stop("Dependent model failed for ", target_pair)
}

ll_ind <- fx$logLik + fy$logLik
ll_dep <- fz$logLik
k_ind <- 2
k_dep <- 12
aic_ind <- -2 * ll_ind + 2 * k_ind
aic_dep <- -2 * ll_dep + 2 * k_dep
lr <- 2 * (ll_dep - ll_ind)
pv <- pchisq(max(0, lr), df = 10, lower.tail = FALSE)
delta <- aic_ind - aic_dep

row <- tibble(pair = target_pair, level = p$level, label = p$focal_label, reference = p$reference,
              n = length(p$x), focal_n = sum(p$x == 1),
              logLik_independent = ll_ind, logLik_dependent = ll_dep,
              AIC_independent = aic_ind, AIC_dependent = aic_dep,
              delta_AIC = delta, likelihood_ratio = lr, df = 10,
              P = pv, fit_ok = TRUE,
              bootstrap_P = NA_real_, bootstrap_nsim = NA_real_, bootstrap_n_valid = NA_real_)

Qdep <- build_Q(fz, levels(z))
rates <- extract_rates(Qdep, target_pair)
saveRDS(rates, rate_path)
write_tsv(rates, file.path(out_pair_dir, paste0(safe, ".rates.tsv")))
saveRDS(list(fx = fx, fy = fy, fz = fz, Qdep = Qdep), fit_path)

n_boot <- as.integer(Sys.getenv("BOOTSTRAP_REPLICATES", unset = cfg$settings$bootstrap_replicates))
if (is.na(n_boot) || n_boot < 1) n_boot <- cfg$settings$bootstrap_replicates
n_cores <- as.integer(Sys.getenv("PBS_NP", unset = "16"))
if (is.na(n_cores) || n_cores < 1) n_cores <- 16
qx <- build_Q(fx, levels(p$x))
qy <- build_Q(fy, levels(p$y))
lr_obs <- lr
cat("BOOTSTRAP_ALL", target_pair, "n_boot=", n_boot, "cores=", n_cores, "\n")
nulls <- unlist(parallel::mclapply(seq_len(n_boot), function(i) {
  sx <- phytools::sim.history(p$tree, qx, message = FALSE)$states
  sy <- phytools::sim.history(p$tree, qy, message = FALSE)$states
  xx <- recode_sim_states(sx, levels(p$x)); names(xx) <- names(sx)
  yy <- recode_sim_states(sy, levels(p$y)); names(yy) <- names(sy)
  zz <- factor(paste0(as.character(xx), "|", as.character(yy))); names(zz) <- names(sx)
  fxi <- tryCatch(phytools::fitMk(p$tree, xx, model = "ER"), error = function(e) NULL)
  fyi <- tryCatch(phytools::fitMk(p$tree, yy, model = "ER"), error = function(e) NULL)
  fzi <- tryCatch(phytools::fitMk(p$tree, zz, model = "ARD"), error = function(e) NULL)
  if (is.null(fxi) || is.null(fyi) || is.null(fzi)) return(NA_real_)
  2 * (fzi$logLik - (fxi$logLik + fyi$logLik))
}, mc.cores = n_cores))
boot_p <- (sum(nulls >= lr_obs, na.rm = TRUE) + 1) / (sum(!is.na(nulls)) + 1)
row$bootstrap_P <- boot_p
row$bootstrap_nsim <- n_boot
row$bootstrap_n_valid <- sum(!is.na(nulls))
saveRDS(nulls, file.path(out_pair_dir, paste0(safe, ".bootstrap_nulls.rds")))

saveRDS(row, row_path)
write_tsv(row, file.path(out_pair_dir, paste0(safe, ".row.tsv")))
cat("DONE", target_pair, "delta_AIC=", delta, "LR=", lr, "P=", pv, "bootstrap_P=", row$bootstrap_P, "\n")



## 16. Pagel correlated-evolution tests within the same cotyledon

In [ ]:
args <- commandArgs(trailingOnly = TRUE)
if (length(args) < 1) stop("usage: Rscript corr_evo_vsall_bootall.R <pair_label>")
target_pair <- args[1]

suppressPackageStartupMessages({
  source(file.path("R", "utils.R"))
  library(dplyr)
  library(readr)
  library(tibble)
  library(phytools)
  library(parallel)
})

cfg <- read_config()
set.seed(cfg$settings$seed)
dat <- load_curated_metadata(cfg)
tree <- load_analysis_tree(cfg)

out_pair_dir <- file.path(cfg$paths$evolutionary, "pair_results")
dir.create(out_pair_dir, recursive = TRUE, showWarnings = FALSE)

dat <- dat %>%
  mutate(
    disaccharide_use = as.integer(disaccharide_use),
    hexose_alcohol_use = as.integer(hexose_alcohol_use),
    source_broad = as.character(source_broad),
    host_major_clade = as.character(host_major_clade),
    host_family_clean = as.character(host_family_clean),
    host_species_clean = as.character(host_species_clean)
  )

source_dat <- dat %>% filter(source_broad %in% c("plant", "soil/rhizosphere", "water"))
plant <- dat %>% filter(source_broad == "plant", !is.na(host_family_clean), !is.na(host_major_clade))
plant_host <- plant %>% filter(!is.na(host_species_clean), host_species_clean != "unknown")

make_pair <- function(dd, eco_var, meta_var, label, level, focal_label, reference) {
  dd <- dd %>% filter(!is.na(.data[[eco_var]]), !is.na(.data[[meta_var]]))
  pr <- prune_to_strains(tree, dd)
  x <- factor(pr$data[[eco_var]])
  y <- factor(pr$data[[meta_var]])
  names(x) <- names(y) <- rownames(pr$data)
  list(x = x, y = y, tree = pr$tree, label = label,
       level = level, focal_label = focal_label, reference = reference)
}

pairs <- list()
add_pair <- function(label, dd, level, focal_label, reference) {
  pairs[[label]] <<- make_pair(dd, "foc", "disaccharide_use", label, level, focal_label, reference)
}

add_pair("Soil_rhizosphere_vs_all_sources_x_disaccharide",
         source_dat %>% mutate(foc = as.integer(source_broad == "soil/rhizosphere")),
         "broad source", "Soil/rhizosphere", "all other sources")
add_pair("Water_vs_all_sources_x_disaccharide",
         source_dat %>% mutate(foc = as.integer(source_broad == "water")),
         "broad source", "Water", "all other sources")
add_pair("Dicotyledon_vs_allplants_x_disaccharide",
         plant %>% mutate(foc = as.integer(host_major_clade == "eudicot")),
         "plant cotyledon", "Dicotyledon", "all other plant hosts")
add_pair("Monocotyledon_vs_allplants_x_disaccharide",
         plant %>% mutate(foc = as.integer(host_major_clade == "monocot")),
         "plant cotyledon", "Monocotyledon", "all other plant hosts")

for (fam in c("Solanaceae", "Musaceae", "Zingiberaceae", "Araceae", "Fabaceae", "Rosaceae", "Cucurbitaceae")) {
  label <- paste0(gsub("[^A-Za-z0-9]+", "_", fam), "_vs_allplants_x_disaccharide")
  add_pair(label, plant %>% mutate(foc = as.integer(host_family_clean == fam)),
           "family", fam, "all other plant hosts")
}


# Family within-clade comparisons: focal family vs other families within the same plant major clade.
for (fam in c("Musaceae", "Araceae", "Zingiberaceae")) {
  label <- paste0(gsub("[^A-Za-z0-9]+", "_", fam), "_vs_monocots_x_disaccharide")
  add_pair(label,
           plant %>% filter(host_major_clade == "monocot") %>% mutate(foc = as.integer(host_family_clean == fam)),
           "family within clade", fam, "other monocot plant hosts")
}
for (fam in c("Solanaceae", "Fabaceae", "Rosaceae", "Cucurbitaceae")) {
  label <- paste0(gsub("[^A-Za-z0-9]+", "_", fam), "_vs_eudicots_x_disaccharide")
  add_pair(label,
           plant %>% filter(host_major_clade == "eudicot") %>% mutate(foc = as.integer(host_family_clean == fam)),
           "family within clade", fam, "other eudicot plant hosts")
}
species_map <- c(
  "Solanum_tuberosum" = "Solanum tuberosum",
  "Capsicum_annuum" = "Capsicum annuum",
  "Curcuma_spp" = "Curcuma spp.",
  "Musa_spp" = "Musa spp.",
  "Nicotiana_tabacum" = "Nicotiana tabacum",
  "Rosa_spp" = "Rosa spp.",
  "Solanum_dulcamara" = "Solanum dulcamara",
  "Solanum_lycopersicum" = "Solanum lycopersicum",
  "Solanum_melongena" = "Solanum melongena",
  "Solanum_virginianum" = "Solanum virginianum",
  "Zingiber_officinale" = "Zingiber officinale"
)
for (nm in names(species_map)) {
  sp <- unname(species_map[[nm]])
  label <- paste0(nm, "_vs_allplants_x_disaccharide")
  add_pair(label, plant_host %>% mutate(foc = as.integer(host_species_clean == sp)),
           "species", sp, "all other plant hosts")
}

# Species within-clade comparisons: focal species vs other plant hosts
# within the same major clade (monocot or eudicot).
species_clade_map <- c(
  "Solanum_tuberosum" = "eudicot",
  "Capsicum_annuum" = "eudicot",
  "Curcuma_spp" = "monocot",
  "Musa_spp" = "monocot",
  "Nicotiana_tabacum" = "eudicot",
  "Rosa_spp" = "eudicot",
  "Solanum_dulcamara" = "eudicot",
  "Solanum_lycopersicum" = "eudicot",
  "Solanum_melongena" = "eudicot",
  "Solanum_virginianum" = "eudicot",
  "Zingiber_officinale" = "monocot"
)
for (nm in names(species_clade_map)) {
  sp <- unname(species_map[[nm]])
  clade <- unname(species_clade_map[[nm]])
  ref <- ifelse(clade == "monocot", "other monocot plant hosts", "other eudicot plant hosts")
  suffix <- ifelse(clade == "monocot", "_vs_monocots_x_disaccharide", "_vs_eudicots_x_disaccharide")
  label <- paste0(nm, suffix)
  add_pair(label,
           plant_host %>% filter(host_major_clade == clade) %>% mutate(foc = as.integer(host_species_clean == sp)),
           "species within clade", sp, ref)
}

if (!target_pair %in% names(pairs)) {
  stop("Unknown pair: ", target_pair, "\nAvailable: ", paste(names(pairs), collapse = ", "))
}
p <- pairs[[target_pair]]

build_Q <- function(fit, state_levels = NULL) {
  k <- nrow(fit$index.matrix)
  Q <- matrix(0, k, k)
  idx <- fit$index.matrix
  ii <- which(idx > 0, arr.ind = TRUE)
  Q[ii] <- fit$rates[idx[ii]]
  diag(Q) <- -rowSums(Q)
  dimnames(Q) <- dimnames(idx)
  if (is.null(rownames(Q)) && !is.null(state_levels) && length(state_levels) == k) {
    rownames(Q) <- colnames(Q) <- state_levels
  }
  Q
}

recode_sim_states <- function(states, state_levels) {
  out <- as.character(states)
  if (all(out %in% state_levels)) {
    return(factor(out, levels = state_levels))
  }
  idx <- suppressWarnings(as.integer(out))
  use_idx <- !is.na(idx) & idx >= 1 & idx <= length(state_levels)
  out[use_idx] <- state_levels[idx[use_idx]]
  factor(out, levels = state_levels)
}

extract_rates <- function(Q, pair) {
  st <- rownames(Q)
  if (!all(c("0|0", "0|1", "1|0", "1|1") %in% st)) {
    return(tibble(pair = pair,
      q_gain_focal = NA_real_, q_gain_reference = NA_real_,
      q_loss_focal = NA_real_, q_loss_reference = NA_real_,
      q_habitat_gain_trait_pos = NA_real_, q_habitat_gain_trait_neg = NA_real_,
      q_habitat_loss_trait_pos = NA_real_, q_habitat_loss_trait_neg = NA_real_,
      metabolic_gain_rate_ratio = NA_real_, metabolic_loss_rate_ratio = NA_real_,
      ecological_transition_rate_ratio = NA_real_))
  }
  get <- function(i, j) Q[i, j]
  q_gain_focal <- get("1|0", "1|1")
  q_gain_reference <- get("0|0", "0|1")
  q_loss_focal <- get("1|1", "1|0")
  q_loss_reference <- get("0|1", "0|0")
  q_hab_gain_pos <- get("0|1", "1|1")
  q_hab_gain_neg <- get("0|0", "1|0")
  q_hab_loss_pos <- get("1|1", "0|1")
  q_hab_loss_neg <- get("1|0", "0|0")
  tibble(pair = pair,
    q_gain_focal = q_gain_focal, q_gain_reference = q_gain_reference,
    q_loss_focal = q_loss_focal, q_loss_reference = q_loss_reference,
    q_habitat_gain_trait_pos = q_hab_gain_pos,
    q_habitat_gain_trait_neg = q_hab_gain_neg,
    q_habitat_loss_trait_pos = q_hab_loss_pos,
    q_habitat_loss_trait_neg = q_hab_loss_neg,
    metabolic_gain_rate_ratio = q_gain_focal / (q_gain_reference + 1e-12),
    metabolic_loss_rate_ratio = q_loss_focal / (q_loss_reference + 1e-12),
    ecological_transition_rate_ratio = q_hab_gain_pos / (q_hab_gain_neg + 1e-12))
}

safe <- gsub("[^A-Za-z0-9_]", "_", target_pair)
row_path <- file.path(out_pair_dir, paste0(safe, ".row.rds"))
rate_path <- file.path(out_pair_dir, paste0(safe, ".rates.rds"))
fit_path <- file.path(out_pair_dir, paste0(safe, ".fits.rds"))

cat("START", target_pair, "n=", length(p$x), "focal_n=", sum(p$x == 1), "\n")

fx <- tryCatch(phytools::fitMk(p$tree, p$x, model = "ER"), error = function(e) e)
fy <- tryCatch(phytools::fitMk(p$tree, p$y, model = "ER"), error = function(e) e)
if (inherits(fx, "error") || inherits(fy, "error")) {
  row <- tibble(pair = target_pair, level = p$level, label = p$focal_label, reference = p$reference,
                n = length(p$x), focal_n = sum(p$x == 1),
                logLik_independent = NA_real_, logLik_dependent = NA_real_,
                AIC_independent = NA_real_, AIC_dependent = NA_real_,
                delta_AIC = NA_real_, likelihood_ratio = NA_real_, df = 10, P = NA_real_, fit_ok = FALSE,
                bootstrap_P = NA_real_, bootstrap_nsim = NA_real_, bootstrap_n_valid = NA_real_)
  saveRDS(row, row_path); write_tsv(row, file.path(out_pair_dir, paste0(safe, ".row.tsv")))
  stop("Independent model failed for ", target_pair)
}

z <- factor(paste0(p$x, "|", p$y))
names(z) <- names(p$x)
fz <- tryCatch(phytools::fitMk(p$tree, z, model = "ARD"), error = function(e) e)
if (inherits(fz, "error")) {
  row <- tibble(pair = target_pair, level = p$level, label = p$focal_label, reference = p$reference,
                n = length(p$x), focal_n = sum(p$x == 1),
                logLik_independent = fx$logLik + fy$logLik, logLik_dependent = NA_real_,
                AIC_independent = NA_real_, AIC_dependent = NA_real_, delta_AIC = NA_real_,
                likelihood_ratio = NA_real_, df = 10, P = NA_real_, fit_ok = FALSE,
                bootstrap_P = NA_real_, bootstrap_nsim = NA_real_, bootstrap_n_valid = NA_real_)
  saveRDS(row, row_path); write_tsv(row, file.path(out_pair_dir, paste0(safe, ".row.tsv")))
  stop("Dependent model failed for ", target_pair)
}

ll_ind <- fx$logLik + fy$logLik
ll_dep <- fz$logLik
k_ind <- 2
k_dep <- 12
aic_ind <- -2 * ll_ind + 2 * k_ind
aic_dep <- -2 * ll_dep + 2 * k_dep
lr <- 2 * (ll_dep - ll_ind)
pv <- pchisq(max(0, lr), df = 10, lower.tail = FALSE)
delta <- aic_ind - aic_dep

row <- tibble(pair = target_pair, level = p$level, label = p$focal_label, reference = p$reference,
              n = length(p$x), focal_n = sum(p$x == 1),
              logLik_independent = ll_ind, logLik_dependent = ll_dep,
              AIC_independent = aic_ind, AIC_dependent = aic_dep,
              delta_AIC = delta, likelihood_ratio = lr, df = 10,
              P = pv, fit_ok = TRUE,
              bootstrap_P = NA_real_, bootstrap_nsim = NA_real_, bootstrap_n_valid = NA_real_)

Qdep <- build_Q(fz, levels(z))
rates <- extract_rates(Qdep, target_pair)
saveRDS(rates, rate_path)
write_tsv(rates, file.path(out_pair_dir, paste0(safe, ".rates.tsv")))
saveRDS(list(fx = fx, fy = fy, fz = fz, Qdep = Qdep), fit_path)

n_boot <- as.integer(Sys.getenv("BOOTSTRAP_REPLICATES", unset = cfg$settings$bootstrap_replicates))
if (is.na(n_boot) || n_boot < 1) n_boot <- cfg$settings$bootstrap_replicates
n_cores <- as.integer(Sys.getenv("PBS_NP", unset = "16"))
if (is.na(n_cores) || n_cores < 1) n_cores <- 16
qx <- build_Q(fx, levels(p$x))
qy <- build_Q(fy, levels(p$y))
lr_obs <- lr
cat("BOOTSTRAP_ALL", target_pair, "n_boot=", n_boot, "cores=", n_cores, "\n")
nulls <- unlist(parallel::mclapply(seq_len(n_boot), function(i) {
  sx <- phytools::sim.history(p$tree, qx, message = FALSE)$states
  sy <- phytools::sim.history(p$tree, qy, message = FALSE)$states
  xx <- recode_sim_states(sx, levels(p$x)); names(xx) <- names(sx)
  yy <- recode_sim_states(sy, levels(p$y)); names(yy) <- names(sy)
  zz <- factor(paste0(as.character(xx), "|", as.character(yy))); names(zz) <- names(sx)
  fxi <- tryCatch(phytools::fitMk(p$tree, xx, model = "ER"), error = function(e) NULL)
  fyi <- tryCatch(phytools::fitMk(p$tree, yy, model = "ER"), error = function(e) NULL)
  fzi <- tryCatch(phytools::fitMk(p$tree, zz, model = "ARD"), error = function(e) NULL)
  if (is.null(fxi) || is.null(fyi) || is.null(fzi)) return(NA_real_)
  2 * (fzi$logLik - (fxi$logLik + fyi$logLik))
}, mc.cores = n_cores))
boot_p <- (sum(nulls >= lr_obs, na.rm = TRUE) + 1) / (sum(!is.na(nulls)) + 1)
row$bootstrap_P <- boot_p
row$bootstrap_nsim <- n_boot
row$bootstrap_n_valid <- sum(!is.na(nulls))
saveRDS(nulls, file.path(out_pair_dir, paste0(safe, ".bootstrap_nulls.rds")))

saveRDS(row, row_path)
write_tsv(row, file.path(out_pair_dir, paste0(safe, ".row.tsv")))
cat("DONE", target_pair, "delta_AIC=", delta, "LR=", lr, "P=", pv, "bootstrap_P=", row$bootstrap_P, "\n")

